# ir_calendar_build_supplier_report — 供應商「財報＋法說整合報告」（HTML）

- 用途：對 `ir_calendar_parse_documents` 解析完的供應商財報 / 法說文件，依採購端新版面（`template/20261021_SCM財報和法說報告html格式v20261007.html`，3M 範例、AUO 配色）產出一家公司一期的完整 HTML 報告：
  給 AUO 採購主管的結論（一段敘事 + 已證實事實 / 管理層展望 / 四面向採購建議）→ 財報資訊 01–06（KPI 四卡、財報實績、事業群表、五年 / 五季營收 × OPM 圖、近五年 11 項指標表 + 十張單欄圖）→ 法說會資訊 01–07（六主題 + Q&A，另 08 採購觀察）→ 資料來源與限制。
- 輸入（silver，全部由 `consume_batches` / `parse_documents` 維護，本 job 不改）：
  `s_{domain}_document_text`（文件全文）、`s_{domain}_document_file`（來源網址、財報公布日 `doc_date`）、`s_{domain}_financials`（年度 + 季別財報數據，含 2026-10-21 新欄：速動比、應收／應付天數、母公司淨利率）、
  `s_{domain}_summary`（爬蟲端法說彙整：`management_views` 六主題 / `business_segments` / `qa_highlights` / `reported_period` **直接進報告**，其餘當參考）、`s_{domain}_company`、`s_{domain}_conference`（法說日期）。
- 輸出：
  - bronze `b_{domain}_supplier_report_llm`：每次 LLM 呼叫一列（原始回應 `payload`、tokens、狀態），append-only
  - gold `g_{domain}_supplier_report`：一家公司一期一列（`html` 全文、`llm_json` 結構化內容、版本、血緣；`light` / `score` 自新版面起一律 NULL），MERGE
  - Volume `<volume_root>/ir_calendar/reports/<分類小寫>/<公司代稱>/<代稱>_<期別>_supplier_report.html`（可直接開啟 / 寄送）
- 數字政策（使用者說明 四）：表格與圖表只用 `s_financials`；LLM 只寫文字、引用數字要標來源；缺值「未取得」、文件沒提「未提及」；沒有示意值模式。
- 法說七節與事業群表的來源順位：爬蟲 summary 結構化欄位（彙整原文）→ 本 job LLM 讀文件補位 → 「未提及」；每節標示來源。
- LLM：Azure Foundry `claude-sonnet-5`（同 meeting_record / sales_report_mail 的 secret `llm-api-keys/ms-foundry`）。一份報告 1～2 次呼叫、輸入最多 `max_total_chars` 字。
- 參數（widgets）：見 [c01] 與下表。
- 排程：接在 `ir_calendar_parse_documents` 之後（同 job 下一個 task），或每天一次；job cluster。`max_reports` 是費用閘門。
- 負責人 / 更新日期：（填）/ 2026-10-21
- 模板與使用者說明：`template/20261021_SCM財報和法說報告html格式v20261007.html`、`template/USER_GUIDE.md`（含逐條對應與已知限制）；舊版面 `template/20261010_…html` 只留對照。

## 一次執行做什麼

1. **工作清單** [c04] `select_work`：`s_document_text` 裡解析成功的文件，依 分類 / 公司 / 期別 篩選後 groupBy（公司, 期別）；`periods` 空 = 每家只取最新一期。
   扣掉 gold 已有「同 prompt 版、同版型、同一組文件 sha256」的報告（`force_rebuild` 不扣）。取前 `max_reports` 份。
2. **每份報告**（ThreadPool，`concurrency` 份併發）：
   1. [c09] 撈該公司的 financials 全部、該期的 summary（沒有就退用最近一期並標示）/ conference / 文件全文與 `doc_date`，collect 到 driver（都是小量）
   2. [c03] `build_facts`：換算單位（台股仟元 → 億、美股元 → 百萬美元）、五年 / 五季數列、實績表（本季 vs 去年同季 + 同比變動）、KPI、11 項指標可得性、summary 結構化欄位整理
   3. [c05] 依 `DOC_KIND_PRIORITY` 排文件、逐檔截尾、總量封頂 → user prompt；[c08] 呼叫 LLM；JSON 壞再問一次、截斷改精簡重打
   4. [c05] `normalize_report`：補齊固定鍵、四個採購面向排序、事業群六欄、法說六主題 + qa
   5. [c07] `render_report`：套新版面 CSS，[c06] 產 inline SVG 圖（標籤碰撞避讓、圖例在圖面上方）；法說七節 / 事業群表先用爬蟲彙整、缺的才用 LLM
   6. [c10] 寫 Volume HTML → gold MERGE；每次 LLM 呼叫都寫 bronze
3. 印每份結果與 tokens；有失敗且 `fail_on_error = true` 就 `raise`。

**重跑安全**：gold 以 (company_key, period) MERGE、Volume 同名覆蓋；bronze 只 append（保留每次呼叫）。同一期文件換新版本（sha256 變）會自動重做。

**只換版型**：改 [c06] / [c07] 後把 [c02] `TEMPLATE_VERSION` 升版，`render_only = true` 跑一次：不呼叫 LLM，用 gold 的 `llm_json` 重套。
（2026-10-21 從 20261010 換到 20261021 同時改了 prompt 結構，舊 `llm_json` 形狀不同：`render_only` 只能重套 prompt 2.0 之後產的報告。）

## 參數

| widget | 預設 | 說明 |
|---|---|---|
| `catalog` / `schema` | `micenter` / `mi3_datahub_prod` | Delta table 所在；對照 `config/project.yml` |
| `domain` | `ir_calendar` | 表名中段 |
| `volume_root` | `/Volumes/micenter/…/unstructured_data_file` | 報告 HTML 落 `<volume_root>/ir_calendar/reports/` |
| `categories` | `SUPPLIER` | 逗號分隔分類；空 = 全部（客戶 / 同業也能產，但 prompt 是採購視角） |
| `company_keys` | 空 | 逗號分隔 `company_key`；空 = 範圍內全部 |
| `periods` | 空 | 逗號分隔 `yyyyQn`；空 = 每家最新一期 |
| `max_reports` | `5` | 單次最多幾份（費用閘門） |
| `max_doc_chars` / `max_total_chars` | `40000` / `120000` | 每份文件 / 全部文件餵 LLM 的字元上限 |
| `model` | `claude-sonnet-5` | Azure Foundry 模型名 |
| `prompt_version` | `2.0` | 要等於 [c05] `PROMPT_VERSION_IN_CELL` |
| `concurrency` | `2` | 併發 LLM 請求數 |
| `force_rebuild` | `false` | `true`：已有同版報告也重做（重新計費）；一定搭配 `company_keys` / `periods` |
| `render_only` | `false` | `true`：不呼叫 LLM，用 gold `llm_json` 重套版 |
| `dry_run` | `false` | `true`：只列工作清單、facts 摘要（含速動比有無、彙整六主題 / 事業群 / Q&A 筆數）與 prompt 大小，不呼叫、不寫 |
| `fail_on_error` | `true` | 有失敗就讓 job 失敗 |
| `job_run_id` | 空 | job parameters 填 `{{job.run_id}}` |

## 維護者須知

| 要改什麼 | 改哪裡 |
|---|---|
| 報告章節 / 版型 | [c07]（CSS 來自 `template/`，不改規則只追加）；改完升 [c02] `TEMPLATE_VERSION`，`render_only` 重套 |
| 圖表樣式 / 標籤規則 | [c06]；`tests/test_ir_calendar_build_supplier_report.py` 有標籤不重疊的檢查 |
| LLM 要寫的內容 / 規則 | [c05] `SYSTEM_PROMPT` + 輸出結構；改了升 `PROMPT_VERSION_IN_CELL` 與 widget `prompt_version`；舊報告不會自動重做（`force_rebuild`） |
| 法說七節的主題詞彙 | [c02] `CONF_TOPICS`：要與爬蟲 `summarize_ir` prompt 的 `management_views.topic` 固定詞彙一致；爬蟲改詞彙這裡同步，否則退成 LLM 補位 |
| 指標表多一項 | 先讓 `s_financials` 有欄位（爬蟲端 → `init_tables` [c21] ADD COLUMNS → `consume_batches` [c04]/[c06]），再改 [c02] `INDICATORS` |
| 金額顯示單位 | [c03] `money_scale` |
| 文件優先序 / 截尾 | [c02] `DOC_KIND_PRIORITY`、[c05] `pack_documents` |
| gold 多欄位 | `init_tables` [c19b] DDL `ADD COLUMNS` → [c10] `REPORT_SCHEMA` + `report_row` |
| 只想本機測邏輯 | [c02]～[c07] 沒有 I/O；`tests/test_ir_calendar_build_supplier_report.py` 用假 financials / 假 summary / 假 LLM JSON 跑 facts → render |

## 首次上線順序（2026-10-21 新版面）

1. 跑 `ir_calendar_init_tables`（[c21] 會對既有 `s_financials` / `s_summary` 補 2026-10-21 新欄；[c19a] / [c19b] 兩張表已存在不受影響）
2. `consume_batches` 貼完 [c04] / [c06] 後 `rebuild_silver = true` 跑一次，讓新欄位從 bronze 補算（爬蟲新版整檔重發後才有值）
3. 本 notebook `dry_run = true`、`company_keys` 指一家有 financials 與彙整的台股供應商（例 `TPE:1303`）：看 facts 摘要（速動比有無、彙整六主題筆數）、prompt 大小、落地路徑
4. `dry_run = false`、`max_reports = 1` 跑一次：打開 Volume 上的 HTML 檢查章節 01–07、圖表、未取得 / 未提及的呈現；對 gold 一列
5. 再試一家沒有彙整的公司（奇美 `TPE:1763`）確認法說節走 LLM 補位 / 未提及；再試沒有 financials 的公司（TYO / KRX）確認「未取得」路徑
6. 放大 `max_reports`，排程接在 `parse_documents` 後


In [ ]:
# [c01] params
# 預設值取自 config/project.yml（catalog / schema）；job 執行時由 job parameters 覆蓋。
# widget 建立過後改 code 預設值不會生效，要重設先 dbutils.widgets.removeAll() 再跑本 cell。布林 / 數字都用字串填，這裡統一轉型。
dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.text("domain", "ir_calendar")      # 表名中段；層級前綴 b_ / s_ / g_ 固定，見 docs/conventions.md 2.1
dbutils.widgets.text("volume_root", "/Volumes/micenter/mi3_datahub_prod/micenterfile_ext/unstructured_data_file")  # 報告 HTML 落 <volume_root>/ir_calendar/reports/
dbutils.widgets.text("categories", "SUPPLIER")    # 逗號分隔的分類代碼；預設只做供應商。空 = 全部分類
dbutils.widgets.text("company_keys", "")          # 逗號分隔的 company_key（TPE:3481,NYSE:GLW）；空 = 範圍內全部
dbutils.widgets.text("periods", "")               # 逗號分隔的曆年期別（2026Q2）；空 = 每家公司「有解析文件的最新一期」
dbutils.widgets.text("max_reports", "5")          # 費用閘門：單次最多產幾份報告（每份 1～2 次 LLM 呼叫、輸入可達 10 萬字）
dbutils.widgets.text("max_doc_chars", "40000")    # 每份文件最多餵 LLM 幾個字元（超過截尾並註明）
dbutils.widgets.text("max_total_chars", "120000") # 一份報告所有文件合計上限（依 [c02] DOC_KIND_PRIORITY 先放重要文件）
dbutils.widgets.text("model", "claude-sonnet-5")  # Azure Foundry 上的模型名（同 meeting_record / sales_report_mail）
dbutils.widgets.text("prompt_version", "2.0")     # 要等於 [c05] PROMPT_VERSION_IN_CELL；改 prompt 就升（2.0 = 2026-10-21 新版面）
dbutils.widgets.text("concurrency", "2")          # 同時打幾個 LLM 請求；輸入大，預設保守
dbutils.widgets.text("force_rebuild", "false")    # true：範圍內已有同版報告也重做（會重新計費）
dbutils.widgets.text("render_only", "false")      # true：不呼叫 LLM，用 gold 既有 llm_json 重新套版（改 [c06]/[c07] 版型時用）
dbutils.widgets.text("dry_run", "false")          # true：只列工作清單與 prompt 大小，不呼叫 LLM、不寫表、不寫 Volume
dbutils.widgets.text("fail_on_error", "true")     # true：本次有失敗的報告就讓 job 失敗（通知才會發）
dbutils.widgets.text("job_run_id", "")            # job parameters 填 {{job.run_id}}


def _flag(name: str) -> bool:
    return dbutils.widgets.get(name).strip().lower() == "true"


def _csv(name: str) -> list[str]:
    return [x.strip() for x in dbutils.widgets.get(name).split(",") if x.strip()]


settings = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "domain": dbutils.widgets.get("domain").strip(),
    "volume_root": dbutils.widgets.get("volume_root").strip().rstrip("/"),
    "categories": _csv("categories"),
    "company_keys": _csv("company_keys"),
    "periods": _csv("periods"),
    "max_reports": int(dbutils.widgets.get("max_reports")),
    "max_doc_chars": int(dbutils.widgets.get("max_doc_chars")),
    "max_total_chars": int(dbutils.widgets.get("max_total_chars")),
    "model": dbutils.widgets.get("model").strip(),
    "prompt_version": dbutils.widgets.get("prompt_version").strip(),
    "concurrency": max(1, int(dbutils.widgets.get("concurrency"))),
    "force_rebuild": _flag("force_rebuild"),
    "render_only": _flag("render_only"),
    "dry_run": _flag("dry_run"),
    "fail_on_error": _flag("fail_on_error"),
    "job_run_id": dbutils.widgets.get("job_run_id").strip() or None,
}
assert settings["catalog"] and settings["schema"] and settings["domain"], "catalog / schema / domain 不可為空"
assert settings["volume_root"].startswith("/Volumes/"), "volume_root 必須是 UC Volume 路徑"
assert settings["max_reports"] >= 1 and settings["max_doc_chars"] >= 1000, "max_reports ≥ 1；max_doc_chars ≥ 1000"
assert settings["concurrency"] <= 8, "concurrency ≤ 8"
assert settings["model"] and settings["prompt_version"], "model / prompt_version 不可為空"
if settings["force_rebuild"] and not (settings["company_keys"] or settings["periods"]):
    print("!!! force_rebuild 沒有搭配 company_keys / periods：範圍內最多 max_reports 份報告會重新計費")
print(settings)


In [ ]:
# [c02] imports
# 共用 import 與常數。本 cell 與 [c03]～[c07] 不碰 spark 全域變數 / dbutils / 檔案系統，可被 tests/ 載入。
import html
import json
import math
import re
import time
from datetime import datetime, timezone

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

UTC = timezone.utc  # noqa: UP017 - 本機測試仍是 Python 3.10，不用 datetime.UTC

# 表命名：<層級前綴><domain>_<短名>，層級前綴固定（docs/conventions.md 2.1）
LAYER_PREFIX = {"bronze": "b_", "silver": "s_", "gold": "g_"}
SRC_COMPANY, SRC_DOC_TEXT, SRC_DOC_FILE = "company", "document_text", "document_file"
SRC_SUMMARY, SRC_FINANCIALS, SRC_CONFERENCE = "summary", "financials", "conference"
LLM_TABLE = "supplier_report_llm"     # bronze：每次 LLM 呼叫一列（原始回應 payload），append-only
REPORT_TABLE = "supplier_report"      # gold：一家公司一期一列（最新版 HTML + 結構化 JSON），MERGE
REPORT_DIR = "ir_calendar/reports"    # <volume_root>/ir_calendar/reports/<分類小寫>/<公司代稱>/<代稱>_<期別>_supplier_report.html

# 版型：template/20261021_SCM財報和法說報告html格式v20261007.html（3M 範例、AUO 配色），2026-10-21 取代 20261010 版；[c07] 版型改了就升
TEMPLATE_VERSION = "20261021"
DOC_STATUS_OK = ("success", "partial")
# 餵 LLM 的文件順序：逐字稿 / 簡報最能回答法說七節，先放；財報本體最長，放最後、受 max_total_chars 截尾
DOC_KIND_PRIORITY = ["Earnings_Transcript", "Presentation", "Press_Release", "Financial_Statements"]

# 近五年指標表的 11 項（新版面第 06 節）。source = s_ir_calendar_financials 欄位；fallback = 主欄位為 NULL 時退用的欄位（畫面標註口徑）。
# kind：money（金額，依 money_scale 換算）/ pct（%）/ ratio（倍）/ eps（元）/ days（天）
# 速動比、應收天數、應付天數：2026-10-21 爬蟲新版起 silver 有 quick_ratio / dso_days / dpo_days（交代文件 20261021 §4）；舊批次為 NULL → 未取得。
INDICATORS = [
    {"key": "revenue", "label": "營收", "kind": "money", "source": "revenue"},
    {"key": "gross_margin", "label": "毛利率", "kind": "pct", "source": "gross_margin"},
    {"key": "pretax_margin", "label": "淨利率（稅前）", "kind": "pct", "source": "pretax_margin"},        # NULL 時 [c03] 由 pretax_income / revenue 算
    {"key": "net_margin_parent", "label": "稅後淨利率", "kind": "pct", "source": "net_margin_parent", "fallback": "net_margin"},  # 模板口徑＝歸屬母公司
    {"key": "eps", "label": "EPS", "kind": "eps", "source": "eps"},
    {"key": "operating_cash_flow", "label": "營業現金流", "kind": "money", "source": "operating_cash_flow"},
    {"key": "debt_ratio", "label": "負債比", "kind": "pct", "source": "debt_ratio"},
    {"key": "current_ratio", "label": "流動比", "kind": "ratio", "source": "current_ratio"},
    {"key": "quick_ratio", "label": "速動比", "kind": "ratio", "source": "quick_ratio"},
    {"key": "ar_days", "label": "應收天數", "kind": "days", "source": "dso_days"},
    {"key": "ap_days", "label": "應付天數", "kind": "days", "source": "dpo_days"},
]
INDICATOR_BY_KEY = {i["key"]: i for i in INDICATORS}
N_YEARS, N_QUARTERS = 5, 5

# 法說會七節（新版面 01–07）。前六節對應爬蟲 summary.management_views.topic 的固定詞彙（2026-10-21 起）；07 取 qa_highlights。
# 爬蟲彙整沒有該主題時才用 LLM 從文件補位（[c05]）；兩邊都沒有 → 「未提及」。
CONF_TOPICS = [
    ("business_lines", "事業體／產品線重點", "01　事業體／產品線重點"),
    ("capacity", "擴產／產能變化", "02　擴產／產能變化"),
    ("profitability", "獲利能力變化", "03　獲利能力變化"),
    ("market_outlook", "未來市場預測", "04　未來市場預測"),
    ("investment_ma_org", "投資／M&A／組織整改", "05　投資／M&A／組織整改"),
    ("auo_topics", "AUO 相關議題", "06　友達4支箭與 Micro LED 相關議題"),
]
CONF_TOPIC_BY_NAME = {name: key for key, name, _ in CONF_TOPICS}
QA_TITLE = "07　Q&A 重點"

ACTION_AREAS = ["供應穩定性", "成本與議價", "付款條件", "替代料或驗證風險"]
NOT_MENTIONED, NOT_AVAILABLE = "未提及", "未取得"

# 版型配色（AUO）：藍 / 淺藍 / 紫 / 黃 / 綠；柱填色與格線
C_BLUE, C_LBLUE, C_PURPLE, C_YELLOW, C_GREEN, C_INK = "#0066B3", "#26A9E0", "#7752A8", "#F5C400", "#40A86B", "#111111"


In [ ]:
# [c03] pure_facts
# 純 Python：把 silver 撈出來的小量列（一家公司的 financials 年度 + 季別、法說彙整、場次、文件清單）整理成報告要用的 facts。
# 數字只來自 s_ir_calendar_financials；算不出來就是 None，畫面顯示「未取得」，絕不補值（使用者說明 四）。
# 法說七節 / 事業群表優先用爬蟲 summary 的結構化欄位（management_views / business_segments / qa_highlights），LLM 只補位；合併在 [c07]。
# 這裡刻意用 list[dict] 不用 DataFrame：一家公司最多幾十列，collect 到 driver 比 Spark 好測、好讀。


def money_scale(currency: str | None, unit: str | None) -> dict:
    """原始金額 → 報告顯示單位。回 {divisor, label, prefix, note}。
    台股 MOPS 仟元 → 億；美股 SEC 元 → 百萬美元；其他幣別以「原始單位 → 億 / 百萬」保守處理並在 note 說明。"""
    cur = (currency or "").upper()
    base = {"thousand": 1e3, "one": 1.0, "million": 1e6}.get((unit or "one").lower(), 1.0)
    if cur == "TWD":
        return {"divisor": 1e8 / base, "label": "億", "prefix": "NT$", "note": "台股 MOPS 仟元換算為億元"}
    if cur == "USD":
        return {"divisor": 1e6 / base, "label": "百萬", "prefix": "US$", "note": "美股 SEC 元換算為百萬美元"}
    if cur in ("JPY", "KRW", "CNY", "HKD"):
        return {"divisor": 1e8 / base, "label": "億", "prefix": cur, "note": f"{cur} 換算為億 {cur}"}
    return {"divisor": 1e6 / base, "label": "百萬", "prefix": cur or "", "note": f"{cur or '未知幣別'} 換算為百萬"}


def to_num(v) -> float | None:
    if v is None:
        return None
    try:
        f = float(v)
    except (TypeError, ValueError):
        return None
    return None if math.isnan(f) or math.isinf(f) else f


def fmt_value(v: float | None, kind: str, *, signed: bool = False) -> str:
    """畫面用的數字字串；None → 未取得。money 已換算；pct 帶 %；ratio 帶 x；eps 兩位小數；days 一位小數。"""
    if v is None:
        return NOT_AVAILABLE
    if kind == "pct":
        return f"{v:+.2f}%" if signed else f"{v:.2f}%"
    if kind == "ratio":
        return f"{v:.2f}x"
    if kind == "days":
        return f"{v:.1f}"
    if kind == "eps":
        return f"{v:+.2f}" if signed else f"{v:.2f}"
    return f"{v:+,.2f}" if signed else f"{v:,.2f}"


def fmt_delta(cur: float | None, prev: float | None, kind: str) -> str:
    """同比變動：money / eps 用 %，pct 用百分點，ratio 用倍數差，days 用天數差；缺值或基期為負（%）→ 未取得 / 基期為負。"""
    if cur is None or prev is None:
        return NOT_AVAILABLE
    if kind in ("money", "eps"):
        if prev == 0:
            return "—"
        if prev < 0:
            return "轉正（基期為負）" if cur >= 0 else "仍為負（基期為負）"
        return f"{(cur / prev - 1) * 100:+.1f}%"
    if kind == "pct":
        return f"{cur - prev:+.2f} pt"
    if kind == "ratio":
        return f"{cur - prev:+.2f}x"
    return f"{cur - prev:+.1f}"


def period_parts(period: str) -> tuple[int, int | None]:
    """'2026Q2' → (2026, 2)；'2025FY' → (2025, None)。格式不合回 (0, None)。"""
    m = re.fullmatch(r"(\d{4})(?:Q([1-4])|FY)", period or "")
    if not m:
        return 0, None
    return int(m.group(1)), (int(m.group(2)) if m.group(2) else None)


def period_label(period: str) -> str:
    y, q = period_parts(period)
    return f"{y} Q{q}" if q else str(y)


def prev_quarter(period: str, n: int = 1) -> str:
    y, q = period_parts(period)
    if not q:
        return ""
    idx = y * 4 + (q - 1) - n
    return f"{idx // 4}Q{idx % 4 + 1}"


def indicator_values(row: dict, scale: dict) -> dict:
    """一列 financials → {indicator_key: 顯示單位的值或 None}。
    pretax_margin 欄位為 NULL 時由 pretax_income / revenue 推算；稅後淨利率主欄 net_margin_parent 缺時退用 net_margin，
    並在 _fallback 記下（畫面標「合併口徑」）。"""
    out = {"_fallback": []}
    for ind in INDICATORS:
        src = ind["source"]
        v = to_num(row.get(src))
        if v is None and ind["key"] == "pretax_margin":
            pre, rev = to_num(row.get("pretax_income")), to_num(row.get("revenue"))
            v = (pre / rev * 100.0) if (pre is not None and rev) else None
        if v is None and ind.get("fallback"):
            v = to_num(row.get(ind["fallback"]))
            if v is not None:
                out["_fallback"].append(ind["key"])
        if ind["kind"] == "money" and v is not None:
            v = v / scale["divisor"]
        out[ind["key"]] = v
    # 實績表 / KPI 另外要的金額欄
    for col in ("net_income", "operating_income", "total_assets", "total_liabilities", "capex", "operating_cash_flow_ytd"):
        v = to_num(row.get(col))
        out[col] = (v / scale["divisor"]) if v is not None else None
    out["operating_margin"] = to_num(row.get("operating_margin"))
    out["net_margin"] = to_num(row.get("net_margin"))
    out["derived"] = list(row.get("derived") or [])
    return out


def _latest_rows(rows: list[dict]) -> dict[str, dict]:
    """同 period 多列（理論上 silver 已 MERGE 唯一）取 seq 最大。回 {period: row}。"""
    out: dict[str, dict] = {}
    for r in rows:
        p = r.get("period")
        if not p:
            continue
        if p not in out or (r.get("seq") or 0) >= (out[p].get("seq") or 0):
            out[p] = r
    return out


def build_annual_series(fin_rows: list[dict], scale: dict, *, n: int = N_YEARS) -> list[dict]:
    """年度列（period_type = annual）最近 n 年，由舊到新。每項 {period, label, values}。"""
    annual = _latest_rows([r for r in fin_rows if r.get("period_type") == "annual"])
    keys = sorted(annual, key=lambda p: period_parts(p)[0])[-n:]
    return [{"period": p, "label": period_label(p), "values": indicator_values(annual[p], scale)} for p in keys]


def build_quarter_series(fin_rows: list[dict], scale: dict, period: str, *, n: int = N_QUARTERS) -> list[dict]:
    """季別列到 period 為止（含）的最近 n 季，由舊到新；中間缺季就缺，不補。"""
    quarterly = _latest_rows([r for r in fin_rows if r.get("period_type") == "quarterly"])
    y, q = period_parts(period)
    if not q:
        return []
    wanted = [prev_quarter(period, i) for i in range(n - 1, -1, -1)]
    return [{"period": p, "label": period_label(p), "values": indicator_values(quarterly[p], scale)}
            for p in wanted if p in quarterly]


ACTUAL_METRICS = [  # 新版面第 02 節：本季 vs 去年同季 + 同比變動
    ("revenue", "營收（{unit}）", "money"), ("operating_margin", "營業利益率", "pct"), ("eps", "每股盈餘（元）", "eps"),
    ("operating_cash_flow", "營業現金流（{unit}，單季）", "money"), ("current_ratio", "流動比（倍）", "ratio"),
    ("quick_ratio", "速動比（倍）", "ratio"),
]


def build_actuals(fin_rows: list[dict], scale: dict, period: str) -> dict:
    """財報實績：本季 vs 去年同季。回 {columns:[{period,label,values}|None ×2], metrics:[{key,label,kind}]}。
    季別 operating_cash_flow 在新版爬蟲為單季推算值；舊批次為 NULL → 未取得（累計值另在 operating_cash_flow_ytd）。"""
    quarterly = _latest_rows([r for r in fin_rows if r.get("period_type") == "quarterly"])
    cols = []
    for p, tag in ((period, "本季"), (prev_quarter(period, 4), "去年同季")):
        r = quarterly.get(p)
        cols.append({"period": p, "label": f"{period_label(p)}" if p else tag, "tag": tag,
                     "values": indicator_values(r, scale) if r else None})
    unit = f"{scale['prefix']} {scale['label']}"
    return {"columns": cols, "metrics": [{"key": k, "label": lb.format(unit=unit), "kind": kd} for k, lb, kd in ACTUAL_METRICS]}


def build_kpis(actuals: dict, scale: dict) -> list[dict]:
    """第 01 節前兩格 KPI：營收 + YoY、營業利益率 + YoY（百分點）。後兩格是事業群營收，由 [c07] 依事業群表補。"""
    cur, yoy = actuals["columns"]
    cv = cur["values"] or {}
    yv = yoy["values"] or {}
    rev, rev_y = cv.get("revenue"), yv.get("revenue")
    opm, opm_y = cv.get("operating_margin"), yv.get("operating_margin")
    yoy_pct = ((rev - rev_y) / abs(rev_y) * 100.0) if (rev is not None and rev_y) else None
    return [
        {"label": f"{period_label(cur['period'])} 營收",
         "value": f"{scale['prefix']} {fmt_value(rev, 'money')} {scale['label']}" if rev is not None else NOT_AVAILABLE,
         "note": (f"YoY {fmt_value(yoy_pct, 'pct', signed=True)}（{period_label(yoy['period'])}：{fmt_value(rev_y, 'money')} {scale['label']}）"
                  if yoy_pct is not None else "YoY 未取得"),
         "tone": "red" if (yoy_pct is not None and yoy_pct < 0) else ("green" if yoy_pct is not None else "")},
        {"label": "營業利益率", "value": fmt_value(opm, "pct"),
         "note": (f"YoY {fmt_delta(opm, opm_y, 'pct')}（{period_label(yoy['period'])}：{fmt_value(opm_y, 'pct')}）"
                  if (opm is not None and opm_y is not None) else f"去年同季：{fmt_value(opm_y, 'pct')}"),
         "tone": "green" if (opm is not None and opm_y is not None and opm >= opm_y) else ("red" if (opm is not None and opm_y is not None) else "")},
    ]


def indicator_availability(annual: list[dict]) -> dict[str, bool]:
    """十一項各自在五年數列裡有沒有至少一個值。"""
    return {ind["key"]: any(s["values"].get(ind["key"]) is not None for s in annual) for ind in INDICATORS}


def _clean_list(v) -> list:
    return [x for x in (v or []) if x is not None]


def summary_segments(summary: dict | None) -> list[dict]:
    """爬蟲 summary.business_segments → [{segment, revenue, revenue_yoy, operating_margin, opm_yoy, note}]（全字串 / None）。"""
    out = []
    for s in _clean_list((summary or {}).get("business_segments")):
        if not isinstance(s, dict) or not (s.get("segment") or "").strip():
            continue
        out.append({k: ((s.get(k) or "").strip() or None) for k in ("segment", "revenue", "revenue_yoy", "operating_margin", "opm_yoy", "note")})
    return out


def summary_views(summary: dict | None) -> dict[str, str]:
    """爬蟲 summary.management_views → {topic_key: view}，只收新版面六類的 topic；舊七類詞彙不對應、略過（留在 other）。"""
    out, other = {}, []
    for v in _clean_list((summary or {}).get("management_views")):
        if not isinstance(v, dict) or not (v.get("view") or "").strip():
            continue
        key = CONF_TOPIC_BY_NAME.get((v.get("topic") or "").strip())
        if key and key not in out:
            out[key] = v["view"].strip()
        elif not key:
            other.append(f"{(v.get('topic') or '其他').strip()}：{v['view'].strip()}")
    if other:
        out["_other"] = other
    return out


def summary_qa(summary: dict | None) -> list[dict]:
    """爬蟲 summary.qa_highlights → [{q, a}]。"""
    out = []
    for x in _clean_list((summary or {}).get("qa_highlights")):
        if isinstance(x, dict) and (x.get("question") or "").strip():
            out.append({"q": x["question"].strip(), "a": (x.get("answer") or "").strip()})
    return out


def build_facts(company: dict, period: str, fin_rows: list[dict], summary: dict | None,
                conference: dict | None, docs: list[dict]) -> dict:
    """報告資料包：[c05] 拿去組 prompt、[c07] 拿去套版。docs 每項至少有 file_name / doc_kind / sha256 / source_url / text_chars / doc_date。
    summary 可能不是同一期（[c09] 找不到同期時退用最近一期），summary_period 記下來源期別。"""
    sample = next((r for r in fin_rows if r.get("currency")), None)
    scale = money_scale(sample.get("currency"), sample.get("unit")) if sample else money_scale(None, None)
    annual = build_annual_series(fin_rows, scale)
    quarterly = build_quarter_series(fin_rows, scale, period)
    actuals = build_actuals(fin_rows, scale, period)
    sources = sorted({u for r in fin_rows for u in (r.get("source_urls") or []) if u})
    sm = summary or {}
    conf_date = (conference or {}).get("conference_date") or sm.get("conference_date")
    report_date = next((d.get("doc_date") for d in docs if d.get("doc_kind") == "Financial_Statements" and d.get("doc_date")), None)
    fallback_years = sorted({s["period"] for s in annual if "net_margin_parent" in s["values"].get("_fallback", [])})
    return {
        "company": {k: company.get(k) for k in ("company_key", "company_name", "english_name", "stock_code",
                                                  "market", "category", "company_slug", "industry", "ir_url")},
        "period": period,
        "period_label": period_label(period),
        "conference_date": str(conf_date) if conf_date else None,
        "report_date": str(report_date) if report_date else None,
        "reported_period": (sm.get("reported_period") or "").strip() or None,
        "summary_period": sm.get("period"),
        "scale": scale,
        "has_financials": bool(fin_rows),
        "financial_source": sorted({r.get("source") for r in fin_rows if r.get("source")}),
        "financial_source_urls": sources,
        "financial_batch": sorted({r.get("batch_id") for r in fin_rows if r.get("batch_id")})[-1:],
        "annual": annual,
        "quarterly": quarterly,
        "actuals": actuals,
        "kpis": build_kpis(actuals, scale),
        "indicator_avail": indicator_availability(annual),
        "net_margin_fallback_years": fallback_years,
        "summary": sm,
        "summary_segments": summary_segments(sm),
        "summary_views": summary_views(sm),
        "summary_qa": summary_qa(sm),
        "docs": [{k: (str(d.get(k)) if k == "doc_date" and d.get(k) else d.get(k))
                  for k in ("file_name", "doc_kind", "sha256", "source_url", "text_chars", "volume_path", "fiscal_label", "doc_date")}
                 for d in docs],
    }


def facts_to_prompt_text(facts: dict) -> str:
    """結構化數據的文字版（給 LLM 看，讓它引用時有依據、不用自己算）。"""
    c, scale = facts["company"], facts["scale"]
    lines = [f"公司：{c.get('company_name')}（{c.get('english_name') or ''}；{c.get('company_key')}；分類 {c.get('category')}；"
             f"產業 {c.get('industry') or '未知'}）", f"報告期別：{facts['period']}（曆年季）",
             f"法說日期：{facts.get('conference_date') or '未知'}；法說報告期間（公司口徑）：{facts.get('reported_period') or '未知'}"]
    if not facts["has_financials"]:
        lines.append("【結構化財報數據】未取得：此公司市場目前沒有結構化財報來源。表格與圖表會顯示「未取得」；"
                     "你在文字中引用任何財報數字時，必須來自下方文件並標明文件名。")
    else:
        lines.append(f"【結構化財報數據】來源 {'/'.join(facts['financial_source'])}；幣別 {scale['prefix']}；"
                     f"金額已換算為「{scale['label']}」（{scale['note']}）；比率為 %。")
        cols = [i for i in INDICATORS if facts["indicator_avail"].get(i["key"])]
        head = "期別 | " + " | ".join(i["label"] for i in cols) + " | 營業利益率"
        lines += ["年度（近五年）：", head]
        for s in facts["annual"]:
            lines.append(f"{s['period']} | " + " | ".join(fmt_value(s["values"].get(i["key"]), i["kind"]) for i in cols)
                         + f" | {fmt_value(s['values'].get('operating_margin'), 'pct')}")
        lines += ["季別（近五季）：", head]
        for s in facts["quarterly"]:
            lines.append(f"{s['period']} | " + " | ".join(fmt_value(s["values"].get(i["key"]), i["kind"]) for i in cols)
                         + f" | {fmt_value(s['values'].get('operating_margin'), 'pct')}")
        missing = [i["label"] for i in INDICATORS if not facts["indicator_avail"].get(i["key"])]
        if missing:
            lines.append("未取得的指標（不要自行推算或引用）：" + "、".join(missing))
    sm = facts.get("summary") or {}
    if sm:
        lines.append(f"【爬蟲端法說彙整（LLM 先前產出，僅供參考，以文件原文為準；期別 {sm.get('period') or '?'}）】")
        for k, lb in (("core_points", "重點"), ("key_numbers", "關鍵數字"), ("risks", "風險"),
                      ("positive_signals", "正面訊號"), ("watch_items", "觀察事項"), ("open_questions", "待釐清")):
            vals = _clean_list(sm.get(k))
            if vals:
                lines.append(f"{lb}：" + "；".join(str(v) for v in vals))
        if sm.get("guidance"):
            lines.append(f"展望：{sm['guidance']}")
        views = facts.get("summary_views") or {}
        for key, name, _ in CONF_TOPICS:
            if views.get(key):
                lines.append(f"經營層觀點［{name}］：{views[key]}")
        for seg in facts.get("summary_segments") or []:
            lines.append(f"事業群［{seg['segment']}］：營收 {seg.get('revenue') or '—'}（YoY {seg.get('revenue_yoy') or '—'}）、"
                         f"營益率 {seg.get('operating_margin') or '—'}（{seg.get('opm_yoy') or '—'}）{('；' + seg['note']) if seg.get('note') else ''}")
        for qa in facts.get("summary_qa") or []:
            lines.append(f"Q&A：{qa['q']} → {qa['a']}")
        covered = [name for key, name, _ in CONF_TOPICS if views.get(key)]
        if covered:
            lines.append("（法說七節中，報告會直接採用上列彙整的主題：" + "、".join(covered) + "；你仍要從文件填寫全部主題，彙整缺的主題會用你的內容）")
    return "\n".join(lines)


In [ ]:
# [c04] spark_transforms
# DataFrame -> DataFrame 純函式：工作清單（哪些 公司 × 期別 要產報告）。無 I/O，可被 tests/ 載入。
WORK_COLS = ["company_key", "company_slug", "category", "period", "doc_count", "doc_kinds", "docs_digest"]


def latest_per_key(df: DataFrame, keys: list[str], order_col: str = "updated_at") -> DataFrame:
    """同鍵多列取 order_col 最大的那列（silver 理論上唯一，這裡保險）。"""
    w = Window.partitionBy(*keys).orderBy(F.col(order_col).desc_nulls_last())
    return df.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")


def candidate_reports(df_text: DataFrame, *, categories: list[str], company_keys: list[str],
                      periods: list[str]) -> DataFrame:
    """s_document_text → 每個 (公司, 期別) 一列：有幾份解析成功的文件、哪些種類、文件版本摘要 docs_digest。
    periods 空 = 每家公司只取「有文件的最新一期」（期別字串 yyyyQn 可直接比大小）。"""
    df = df_text.filter(F.col("parse_status").isin(*DOC_STATUS_OK) & F.col("period").rlike(r"^\d{4}Q[1-4]$"))
    if categories:
        df = df.filter(F.col("category").isin(*categories))
    if company_keys:
        df = df.filter(F.col("company_key").isin(*company_keys))
    if periods:
        df = df.filter(F.col("period").isin(*periods))
    grouped = (df.groupBy("company_key", "company_slug", "category", "period")
               .agg(F.count("*").alias("doc_count"),
                    F.array_sort(F.collect_set("doc_kind")).alias("doc_kinds"),
                    F.sha2(F.concat_ws(",", F.array_sort(F.collect_set("sha256"))), 256).alias("docs_digest")))
    if not periods:
        w = Window.partitionBy("company_key").orderBy(F.col("period").desc())
        grouped = grouped.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")
    return grouped.select(*WORK_COLS)


def select_work(df_text: DataFrame, df_report: DataFrame, *, categories: list[str], company_keys: list[str],
                periods: list[str], prompt_version: str, template_version: str, force: bool) -> DataFrame:
    """候選扣掉 gold 已有「同 prompt 版、同版型、同一組文件」的報告（force = True 不扣）。
    文件有新版本（sha256 變）或 prompt / 版型升版 → docs_digest / 版本不同 → 重做。"""
    cand = candidate_reports(df_text, categories=categories, company_keys=company_keys, periods=periods)
    if force:
        return cand
    done = (df_report.filter((F.col("prompt_version") == F.lit(prompt_version))
                             & (F.col("template_version") == F.lit(template_version)))
            .select("company_key", "period", "docs_digest"))
    return cand.join(done, ["company_key", "period", "docs_digest"], "left_anti").select(*WORK_COLS)


In [ ]:
# [c05] prompt
# LLM 只做「讀文件、寫文字段落」：給 AUO 採購主管的結論（一段敘事 + 已證實事實 / 管理層展望 / 四面向採購建議）、財報重點、
# 事業群表與法說七節的補位（爬蟲 summary 已有的主題報告會優先用彙整原文，[c07] 合併）。
# 數字表格與圖表由 [c03] / [c06] 從結構化數據產生，LLM 不得編數字（使用者說明 四）。prompt 內嵌成常數；改這個 cell 就升 [c01] prompt_version。
# 2.0（2026-10-21）：對齊新版面——拿掉燈號 / 分數 / War Room / Watchlist / QBR；法說六主題改用爬蟲固定詞彙；事業群表改成結構化六欄。
PROMPT_VERSION_IN_CELL = "2.0"   # [c11] 會比對 settings["prompt_version"]，不一致就停
LLM_MAX_TOKENS = 12000
TRUNCATED_MARK = "\n…（本文件超過長度上限，以下截尾）"

SYSTEM_PROMPT = """你是 AUO（友達光電）採購部門的供應商分析師。輸入是一家供應商某一季的公開資料：結構化財報數據（程式已算好）、爬蟲端先前的法說彙整（含經營層觀點、事業群、Q&A）、以及該季的財報 / 法說文件全文（PDF / HTML 解析後的 markdown）。
你的工作是寫出「財報＋法說整合報告」的文字內容，輸出 **單一 JSON 物件**，不加任何說明、不用 markdown 圍欄。

## 寫作規則
1. 繁體中文、採購主管可直接讀的商務語氣；每條 ≤ 80 字；用詞具體（產品線、產能、期程、金額），不寫空話。
2. 嚴格區分三種內容：**已證實事實**（文件或財報數據明載）、**管理層展望**（公司自己的預期 / 指引）、**採購建議**（你的判讀）。不得混寫。
3. 採購結論聚焦四個面向：供應穩定性、成本與議價、付款條件、替代料或驗證風險。**不得**把公司整體財務變動直接推論為個別料號的漲價或降價理由；只能寫成「議價時可留意 / 需向供應商確認」。
4. 引用數字只能來自「結構化財報數據」或文件原文，並在句尾標來源：（財報數據）或（文件名）。不得自行推算、不得補缺值。結構化數據標「未取得」的指標不要引用。
5. 文件沒提到的章節，對應欄位給空陣列 []（畫面會顯示「未提及」）。**不得**推測與 AUO 的合作關係、供應項目或 Micro LED 相關性；只有文件明確提到才寫。
6. 「AUO 相關議題」：只收文件中明確提到 AUO / 友達、面板 / 顯示器客戶、Micro LED、Mini LED、車用顯示、CPO / 光通訊的段落；沒有就 []。
7. Q&A 重點：只有逐字稿或簡報有 Q&A 才填；每題 q 一句、a 一到兩句。
8. 事業群表：只有文件有分部 / 產品線數字才填；值照文件原文字串（含單位與幣別），沒有的欄位給 null；不得用整體數字拆算。
9. 爬蟲端法說彙整只是參考；與文件原文衝突時以原文為準。彙整已有的主題報告會優先採用彙整原文，但你仍要從文件填寫全部主題，彙整缺的主題會用你的內容。
10. conclusion 是給採購主管看的一段敘事（4～6 句）：營收與獲利趨勢、資產負債緩衝、本業獲利與現金流、對 AUO 採購的意義；數字一律標來源。

## 輸出 JSON 結構（鍵名固定，全部必填；沒有內容給 [] 或 ""）
{
  "conclusion": "4～6 句敘事（見規則 10）",
  "confirmed_facts": ["已證實事實 …（來源）"],
  "management_outlook": ["管理層展望 …（來源）"],
  "procurement_actions": [
    {"area": "供應穩定性", "advice": "…"},
    {"area": "成本與議價", "advice": "…"},
    {"area": "付款條件", "advice": "…"},
    {"area": "替代料或驗證風險", "advice": "…"}
  ],
  "financial_highlights": ["財報重點 …（來源）"],
  "segments": [{"segment": "事業群 / 產品線名", "revenue": "營收（原文字串）或 null", "revenue_yoy": "YoY 或 null",
                "operating_margin": "營益率 / 部門利潤（原文字串）或 null", "opm_yoy": "YoY 或 null", "note": "一句說明（來源）"}],
  "conference": {
    "business_lines": ["事業體／產品線重點"],
    "capacity": ["擴產／產能變化"],
    "profitability": ["獲利能力變化"],
    "market_outlook": ["未來市場預測"],
    "investment_ma_org": ["投資／M&A／組織整改"],
    "auo_topics": ["AUO 相關議題（四支箭 / Micro LED / 車用顯示 / CPO）"],
    "qa": [{"q": "問題", "a": "回答重點"}]
  },
  "sources_used": ["實際引用到的文件名"],
  "caveats": ["口徑限制、文件缺漏、你無法判斷的地方"]
}"""

RETRY_HINT = ("你上一輪的輸出無法解析為 JSON：{err}。請只輸出一個合法 JSON 物件（鍵名照規格、字串內的引號要跳脫、"
              "不要 markdown 圍欄、不要前後說明）。")
COMPACT_HINT = ("\n\n（上一輪輸出超過長度上限被截斷。請精簡：每個陣列最多 6 條、每條 ≤ 50 字、qa 最多 6 題，其餘規則不變。）")

REPORT_LIST_KEYS = ["confirmed_facts", "management_outlook", "financial_highlights", "sources_used", "caveats"]
CONF_KEYS = [key for key, _, _ in CONF_TOPICS] + ["qa"]
SEGMENT_KEYS = ["segment", "revenue", "revenue_yoy", "operating_margin", "opm_yoy", "note"]


def pack_documents(docs: list[dict], *, max_doc_chars: int, max_total_chars: int) -> tuple[list[dict], list[str]]:
    """依 DOC_KIND_PRIORITY 排序、逐檔截尾、總量封頂。回 (放進 prompt 的文件 [{file_name, doc_kind, text, truncated}], 被略過的檔名)。"""
    order = {k: i for i, k in enumerate(DOC_KIND_PRIORITY)}
    ranked = sorted(docs, key=lambda d: (order.get(d.get("doc_kind"), len(order)), d.get("file_name") or ""))
    out, skipped, used = [], [], 0
    for d in ranked:
        text = (d.get("text_md") or "").strip()
        if not text:
            skipped.append(d.get("file_name") or "?")
            continue
        room = min(max_doc_chars, max_total_chars - used)
        if room < 2000:
            skipped.append(d.get("file_name") or "?")
            continue
        truncated = len(text) > room
        body = text[:room] + TRUNCATED_MARK if truncated else text
        out.append({"file_name": d.get("file_name"), "doc_kind": d.get("doc_kind"), "text": body, "truncated": truncated})
        used += len(body)
    return out, skipped


def build_user_prompt(facts: dict, packed_docs: list[dict], skipped: list[str]) -> str:
    parts = ["# 結構化資料", facts_to_prompt_text(facts), "", "# 文件全文（markdown；表格以 | 分隔）"]
    if not packed_docs:
        parts.append("（本期沒有可用的文件全文）")
    for d in packed_docs:
        parts += [f"\n## 文件：{d['file_name']}（{d['doc_kind']}{'；已截尾' if d['truncated'] else ''}）", d["text"]]
    if skipped:
        parts.append("\n（因長度上限未放入的文件：" + "、".join(skipped) + "）")
    parts.append("\n請依 system 規格輸出 JSON。")
    return "\n".join(parts)


def parse_llm_json(text: str | None) -> tuple[dict | None, str | None]:
    """容忍 ```json 圍欄與前後雜訊：取第一個 { 到最後一個 }。回 (obj, error)。"""
    if not text or not text.strip():
        return None, "empty output"
    s = text.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s)
    s = re.sub(r"\s*```$", "", s)
    i, j = s.find("{"), s.rfind("}")
    if i < 0 or j <= i:
        return None, "no JSON object found"
    try:
        obj = json.loads(s[i:j + 1])
    except json.JSONDecodeError as e:
        return None, f"JSONDecodeError: {e}"
    if not isinstance(obj, dict):
        return None, "top level is not an object"
    return obj, None


def _str_list(v) -> list[str]:
    if isinstance(v, str):
        return [v.strip()] if v.strip() else []
    if not isinstance(v, list):
        return []
    return [str(x).strip() for x in v if x is not None and str(x).strip()]


def _dict_list(v, keys: list[str], *, keep_none: bool = False) -> list[dict]:
    """list[dict] 收斂成固定鍵；keep_none = True 時缺值保留 None（事業群表要分「沒給」與空字串），否則補空字串。"""
    if not isinstance(v, list):
        return []
    out = []
    for x in v:
        if isinstance(x, dict):
            d = {}
            for k in keys:
                val = x.get(k)
                s = str(val).strip() if val is not None else ""
                d[k] = (s or None) if keep_none else s
            out.append(d)
        elif isinstance(x, str) and x.strip():
            out.append({keys[0]: x.strip(), **{k: (None if keep_none else "") for k in keys[1:]}})
    return [d for d in out if any(d.values())]


def normalize_report(obj: dict | None) -> dict:
    """LLM 輸出 → 固定形狀（缺鍵補空、型別修正、四個採購面向排序），[c07] 才不用到處判 None。"""
    obj = obj or {}
    actions = _dict_list(obj.get("procurement_actions"), ["area", "advice"])
    by_area = {a["area"]: a["advice"] for a in actions if a["advice"]}
    actions = [{"area": a, "advice": by_area.get(a, "")} for a in ACTION_AREAS] + \
              [a for a in actions if a["area"] not in ACTION_AREAS and a["advice"]]
    conf = obj.get("conference") if isinstance(obj.get("conference"), dict) else {}
    segs = [s for s in _dict_list(obj.get("segments"), SEGMENT_KEYS, keep_none=True) if s.get("segment")]
    return {
        "conclusion": str(obj.get("conclusion") or "").strip(),
        "confirmed_facts": _str_list(obj.get("confirmed_facts")),
        "management_outlook": _str_list(obj.get("management_outlook")),
        "procurement_actions": actions,
        "financial_highlights": _str_list(obj.get("financial_highlights")),
        "segments": segs,
        "conference": {**{k: _str_list(conf.get(k)) for k in CONF_KEYS if k != "qa"},
                       "qa": _dict_list(conf.get("qa"), ["q", "a"])},
        "sources_used": _str_list(obj.get("sources_used")),
        "caveats": _str_list(obj.get("caveats")),
    }


In [ ]:
# [c06] charts_svg
# 純 Python 產 inline SVG：淺藍柱（營收 / 指標）＋ 紫色折線（OPM / 趨勢）、黃色標記點。不用 matplotlib（cluster 無中文字型會變豆腐）、
# 不用 JS / CDN（報告要能離線單獨開啟、列印）。版面規則照使用者說明 二：營收數字放柱內中間偏上、所有標籤不互相遮蓋、
# 不被折線 / 圖例遮蓋（圖例放圖面上方獨立列；標籤框碰撞自動避讓；文字加白邊 halo，折線穿過也讀得到）。
# 配色 2026-10-21 換成新版面（AUO）：柱 #cfeafa 淺藍 + 黑框、折線 #7752A8 紫、標記 #F5C400 黃；模板的 12 張 base64 PNG 由這裡的 SVG 取代。
# chart_layout() 先算幾何（含每個標籤的框，tests 用它驗「不重疊」），chart_svg() 只負責畫。
CHART_W, CHART_H = 1100, 380
PLOT = {"l": 72, "r": 72, "t": 48, "b": 44}
BAR_FILL, BAR_STROKE = "#cfeafa", C_INK                 # 淺藍柱、黑框（模板樣式）
LINE_COLOR, MARK_FILL = C_PURPLE, C_YELLOW               # 紫線、黃點
GRID, AXIS_TXT, MUTED = "#e3eef5", "#444444", "#8aa0b5"
FONT = "system-ui,'Noto Sans TC','Microsoft JhengHei',sans-serif"
LABEL_FS, AXIS_FS = 12, 11
LABEL_H = 14


def _nice_step(span: float, divisions: int = 4) -> float:
    raw = span / divisions if span > 0 else 1.0
    mag = 10 ** math.floor(math.log10(raw))
    for m in (1, 2, 2.5, 5, 10):
        if raw <= m * mag:
            return m * mag
    return 10 * mag


def _axis(values: list[float | None], *, headroom: float = 0.18) -> dict:
    """含 0 的軸：回 {min, max, step, ticks}。上方留 headroom 給標籤。"""
    vals = [v for v in values if v is not None]
    lo, hi = min(0.0, *vals) if vals else 0.0, max(0.0, *vals) if vals else 1.0
    if hi == lo:
        hi = lo + 1.0
    span = hi - lo
    hi += span * headroom
    if lo < 0:
        lo -= span * 0.08
    step = _nice_step(hi - lo)
    lo = math.floor(lo / step) * step
    hi = math.ceil(hi / step) * step
    ticks = []
    t = lo
    while t <= hi + step * 1e-9:
        ticks.append(round(t, 10))
        t += step
    return {"min": lo, "max": hi, "step": step, "ticks": ticks}


def _text_w(s: str, fs: float = LABEL_FS) -> float:
    """估文字寬：ASCII 0.58em、CJK 1.0em。只用來算標籤框，不求精準。"""
    return sum((1.0 if ord(ch) > 0x2E7F else 0.58) * fs for ch in s) + 6


def _box(cx: float, y_baseline: float, text: str, fs: float = LABEL_FS) -> dict:
    w = _text_w(text, fs)
    return {"x": cx - w / 2, "y": y_baseline - fs, "w": w, "h": LABEL_H, "text": text, "cx": cx, "ty": y_baseline}


def _overlap(a: dict, b: dict, pad: float = 2.0) -> bool:
    return not (a["x"] + a["w"] + pad <= b["x"] or b["x"] + b["w"] + pad <= a["x"]
                or a["y"] + a["h"] + pad <= b["y"] or b["y"] + b["h"] + pad <= a["y"])


def chart_layout(labels: list[str], bars: list[float | None], line: list[float | None] | None, *,
                 bar_kind: str, line_kind: str = "pct", line_labels: bool = True,
                 shared_axis: bool = False) -> dict:
    """算幾何。bars / line 長度 = labels；None = 缺值（柱不畫、寫「未取得」；折線斷開）。
    shared_axis = True：折線用柱的軸（單一指標圖：柱 + 趨勢線同一數列）。"""
    n = max(1, len(labels))
    px, py = PLOT["l"], PLOT["t"]
    pw, ph = CHART_W - PLOT["l"] - PLOT["r"], CHART_H - PLOT["t"] - PLOT["b"]
    slot = pw / n
    bw = slot * 0.5
    ax_b = _axis(bars)
    ax_l = ax_b if (shared_axis or not line) else _axis(line)

    def yb(v: float) -> float:
        return py + (ax_b["max"] - v) / (ax_b["max"] - ax_b["min"]) * ph

    def yl(v: float) -> float:
        return py + (ax_l["max"] - v) / (ax_l["max"] - ax_l["min"]) * ph

    y0 = yb(0.0)
    bar_rects, bar_labels, missing = [], [], []
    for i, v in enumerate(bars):
        cx = px + slot * (i + 0.5)
        if v is None:
            missing.append(_box(cx, y0 - 6, NOT_AVAILABLE, AXIS_FS))
            continue
        top, bottom = (yb(v), y0) if v >= 0 else (y0, yb(v))
        bar_rects.append({"x": cx - bw / 2, "y": top, "w": bw, "h": max(0.0, bottom - top), "v": v})
        text = fmt_value(v, bar_kind)
        h = bottom - top
        if v >= 0:
            ty = top + 16 if h >= 26 else top - 5          # 柱內中間偏上；柱太矮放柱頂上方
        else:
            ty = bottom + 14                               # 負值放柱底下方
        bar_labels.append(_box(cx, ty, text))

    pts, line_labels_out = [], []
    if line:
        for i, v in enumerate(line):
            if v is None:
                pts.append(None)
                continue
            cx = px + slot * (i + 0.5)
            pts.append({"x": cx, "y": yl(v), "v": v})
    taken = bar_labels + missing
    if line and line_labels:
        for p in pts:
            if p is None:
                continue
            text = fmt_value(p["v"], line_kind, signed=(line_kind == "pct"))
            cands = [p["y"] - 9, p["y"] + 19]
            placed = None
            for ty in cands:
                b = _box(p["x"], ty, text)
                if b["y"] >= py - 2 and b["y"] + b["h"] <= CHART_H - 4 and not any(_overlap(b, t) for t in taken):
                    placed = b
                    break
            if placed is None:                              # 兩個位置都撞：往上找空位
                ty = p["y"] - 9
                for _ in range(24):
                    b = _box(p["x"], ty, text)
                    if not any(_overlap(b, t) for t in taken):
                        placed = b
                        break
                    ty -= 5
                placed = placed or _box(p["x"], p["y"] - 9, text)
            taken.append(placed)
            line_labels_out.append(placed)

    return {
        "w": CHART_W, "h": CHART_H, "plot": {"x": px, "y": py, "w": pw, "h": ph}, "slot": slot, "y0": y0,
        "axis_bar": {**ax_b, "ticks_y": [yb(t) for t in ax_b["ticks"]]},
        "axis_line": None if (not line or shared_axis) else {**ax_l, "ticks_y": [yl(t) for t in ax_l["ticks"]]},
        "labels": labels, "bar_kind": bar_kind, "line_kind": line_kind,
        "bars": bar_rects, "bar_labels": bar_labels, "missing": missing,
        "points": pts, "line_labels": line_labels_out,
    }


def overlapping_labels(layout: dict) -> list[tuple[str, str]]:
    """tests 用：回所有互相重疊的標籤文字對（應為空）。"""
    boxes = layout["bar_labels"] + layout["line_labels"] + layout["missing"]
    out = []
    for i in range(len(boxes)):
        for j in range(i + 1, len(boxes)):
            if _overlap(boxes[i], boxes[j]):
                out.append((boxes[i]["text"], boxes[j]["text"]))
    return out


def _svg_text(x: float, y: float, text: str, *, fs: float = LABEL_FS, fill: str = C_INK, anchor: str = "middle",
              weight: str = "700", halo: bool = True) -> str:
    style = f"font-family:{FONT};font-size:{fs}px;font-weight:{weight};"
    if halo:
        style += "paint-order:stroke;stroke:#ffffff;stroke-width:3px;stroke-linejoin:round;"
    return (f'<text x="{x:.1f}" y="{y:.1f}" text-anchor="{anchor}" fill="{fill}" style="{style}">'
            f"{html.escape(text)}</text>")


def chart_svg(layout: dict, *, bar_name: str, line_name: str | None, bar_unit: str, line_unit: str = "%",
              title: str = "") -> str:
    """layout → <svg>。圖例在最上一列（y≈20），圖面從 PLOT.t 開始，互不重疊。"""
    L = layout
    p = L["plot"]
    out = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {L["w"]} {L["h"]}" width="100%" '
           f'role="img" aria-label="{html.escape(title or bar_name)}" style="display:block;max-width:100%;height:auto">']
    # 圖例列
    lx = p["x"]
    out.append(f'<rect x="{lx}" y="12" width="14" height="14" rx="2" fill="{BAR_FILL}" stroke="{BAR_STROKE}"/>')
    out.append(_svg_text(lx + 20, 24, f"{bar_name}（{bar_unit}）", fs=AXIS_FS, fill=AXIS_TXT, anchor="start", weight="500", halo=False))
    if line_name:
        lx2 = lx + 20 + _text_w(f"{bar_name}（{bar_unit}）", AXIS_FS) + 18
        out.append(f'<line x1="{lx2}" y1="19" x2="{lx2 + 22}" y2="19" stroke="{LINE_COLOR}" stroke-width="2.5"/>')
        out.append(f'<circle cx="{lx2 + 11}" cy="19" r="3.5" fill="{MARK_FILL}" stroke="{LINE_COLOR}" stroke-width="1.5"/>')
        out.append(_svg_text(lx2 + 28, 24, f"{line_name}（{line_unit}）", fs=AXIS_FS, fill=AXIS_TXT, anchor="start", weight="500", halo=False))
    # 格線 + 左軸
    ab = L["axis_bar"]
    for t, y in zip(ab["ticks"], ab["ticks_y"], strict=True):
        out.append(f'<line x1="{p["x"]}" y1="{y:.1f}" x2="{p["x"] + p["w"]}" y2="{y:.1f}" stroke="{GRID}" stroke-width="1"/>')
        out.append(_svg_text(p["x"] - 8, y + 4, _tick_text(t, L["bar_kind"]), fs=AXIS_FS, fill=AXIS_TXT, anchor="end", weight="500", halo=False))
    # 右軸（折線）
    if L["axis_line"]:
        al = L["axis_line"]
        for t, y in zip(al["ticks"], al["ticks_y"], strict=True):
            out.append(_svg_text(p["x"] + p["w"] + 8, y + 4, _tick_text(t, L["line_kind"]), fs=AXIS_FS, fill=LINE_COLOR, anchor="start", weight="500", halo=False))
    # 零線
    out.append(f'<line x1="{p["x"]}" y1="{L["y0"]:.1f}" x2="{p["x"] + p["w"]}" y2="{L["y0"]:.1f}" stroke="{MUTED}" stroke-width="1.2"/>')
    # 柱
    for r in L["bars"]:
        out.append(f'<rect x="{r["x"]:.1f}" y="{r["y"]:.1f}" width="{r["w"]:.1f}" height="{r["h"]:.1f}" '
                   f'fill="{BAR_FILL}" stroke="{BAR_STROKE}" stroke-width="1"/>')
    # 折線 + 點
    pts = L["points"]
    if pts:
        segs, cur = [], []
        for pt in pts:
            if pt is None:
                if cur:
                    segs.append(cur)
                cur = []
            else:
                cur.append(pt)
        if cur:
            segs.append(cur)
        for seg in segs:
            if len(seg) >= 2:
                d = " ".join(f"{'M' if i == 0 else 'L'}{q['x']:.1f},{q['y']:.1f}" for i, q in enumerate(seg))
                out.append(f'<path d="{d}" fill="none" stroke="{LINE_COLOR}" stroke-width="2.6" stroke-linejoin="round"/>')
            for q in seg:
                out.append(f'<circle cx="{q["x"]:.1f}" cy="{q["y"]:.1f}" r="5" fill="{MARK_FILL}" stroke="{LINE_COLOR}" stroke-width="2"/>')
    # 標籤（最後畫，疊在線與柱之上）
    for b in L["bar_labels"]:
        out.append(_svg_text(b["cx"], b["ty"], b["text"], fill=C_INK))
    for b in L["missing"]:
        out.append(_svg_text(b["cx"], b["ty"], b["text"], fs=AXIS_FS, fill=MUTED, weight="500"))
    for b in L["line_labels"]:
        out.append(_svg_text(b["cx"], b["ty"], b["text"], fill=LINE_COLOR))
    # X 軸類別
    for i, lb in enumerate(L["labels"]):
        cx = p["x"] + L["slot"] * (i + 0.5)
        out.append(_svg_text(cx, L["h"] - 14, lb, fs=AXIS_FS, fill=AXIS_TXT, weight="500", halo=False))
    out.append("</svg>")
    return "\n".join(out)


def _tick_text(t: float, kind: str) -> str:
    if kind == "pct":
        return f"{t:g}%"
    if kind == "ratio":
        return f"{t:g}x"
    return f"{t:,.0f}" if abs(t) >= 10 else f"{t:g}"


def svg_revenue_opm(series: list[dict], scale: dict, *, title: str) -> str:
    """營收（柱）× 營業利益率（折線）。series = [c03] build_annual_series / build_quarter_series 的輸出。"""
    labels = [s["label"] for s in series]
    bars = [s["values"].get("revenue") for s in series]
    line = [s["values"].get("operating_margin") for s in series]
    layout = chart_layout(labels, bars, line, bar_kind="money", line_kind="pct")
    return chart_svg(layout, bar_name="營收", line_name="營業利益率 OPM", bar_unit=f"{scale['prefix']} {scale['label']}",
                     line_unit="%", title=title)


def svg_indicator(series: list[dict], key: str, scale: dict) -> str:
    """單一指標五年圖：淺藍柱 + 同數列紫色趨勢線（共用軸、折線不另標數字，避免重複）。"""
    ind = INDICATOR_BY_KEY[key]
    labels = [s["label"] for s in series]
    vals = [s["values"].get(key) for s in series]
    layout = chart_layout(labels, vals, vals, bar_kind=ind["kind"], line_kind=ind["kind"], line_labels=False, shared_axis=True)
    unit = {"money": f"{scale['prefix']} {scale['label']}", "pct": "%", "ratio": "倍", "eps": "元", "days": "天"}[ind["kind"]]
    return chart_svg(layout, bar_name=ind["label"], line_name="趨勢", bar_unit=unit, line_unit="同左", title=ind["label"])


In [ ]:
# [c07] render_html
# facts（[c03]）+ LLM 結構化輸出（[c05] normalize_report）→ 完整可單獨開啟的 HTML。
# 版型：template/20261021_SCM財報和法說報告html格式v20261007.html（3M 範例、AUO 配色）的 <style> 原樣內嵌（TEMPLATE_CSS），
# 章節照模板：頁首 → 給 AUO 採購主管的結論 → 財報資訊 01–06 → 法說會資訊 01–07 → 資料來源與限制。模板的 12 張 PNG 改為 [c06] 的 inline SVG。
# 合併規則：事業群表與法說 01–06 優先用爬蟲 summary 的結構化欄位（business_segments / management_views），缺的主題才用 LLM；
# 07 Q&A 用 qa_highlights，缺時用 LLM。所有文字經 html.escape；數字只從 facts 來；空清單一律印「未提及」，缺值印「未取得」。
TEMPLATE_CSS = """*{box-sizing:border-box}body{margin:0;background:#edf7fc;color:#111111;font:15px/1.7 system-ui,'Noto Sans TC','Microsoft JhengHei',sans-serif}
header{background:linear-gradient(110deg,#0066B3,#26A9E0);color:white;padding:34px max(22px,calc((100vw - 1100px)/2))}h1{margin:0}header p{margin:6px 0 0}
main{max-width:1100px;margin:auto;padding:20px}
.band{background:#7752A8;color:white;padding:12px 20px;border-radius:10px;margin:26px 0 14px;font-size:20px;font-weight:bold}
section{background:white;border:1px solid #cbe7f6;border-radius:15px;padding:22px;margin:15px 0;box-shadow:0 5px 20px #0066b318}
h2{margin:0 0 12px;color:#0066B3;font-size:20px}h3{color:#7752A8}
table{border-collapse:collapse;width:100%;min-width:670px}th,td{text-align:left;border-bottom:1px solid #cbe7f6;padding:9px 11px;vertical-align:top}th{background:#def3fb;color:#0066B3}
.scroll{overflow:auto}
.grid{display:grid;grid-template-columns:repeat(4,1fr);gap:12px}
.kpi{background:#def3fb;border-top:4px solid #26A9E0;border-radius:11px;padding:13px}.kpi:nth-child(2){border-color:#F5C400}.kpi:nth-child(3){border-color:#40A86B}.kpi:nth-child(4){border-color:#7752A8}
.kpi b{display:block;font-size:23px;color:#111111}.kpi small,.mini{font-size:12px;color:#111111}
.note{background:#fff9d9;border-left:4px solid #F5C400;padding:12px;margin:12px 0}
.two{display:grid;grid-template-columns:1fr 1fr;gap:14px}.card{padding:14px;background:#f2eefa;border:1px solid #d9cbed;border-radius:10px}
a{color:#0066B3}footer{padding:15px}
.delta-positive{color:#238451;font-weight:700}.delta-negative{color:#C62828;font-weight:700}
.trend-grid{display:grid;grid-template-columns:minmax(0,1fr);gap:26px;margin-top:26px}
.trend-card{border:1px solid #cbe7f6;border-radius:12px;padding:22px 24px;background:#fff;overflow:hidden}.trend-card h3{margin:0 0 12px;color:#7752A8;font-size:20px}.trend-card small{font-weight:normal;color:#444}
.chart{margin:10px 0 4px}.chart-missing{padding:30px 12px;text-align:center;color:#444;background:#f7fbfe;border:1px dashed #cbe7f6;border-radius:10px}
.na{color:#666}ul,ol{padding-left:20px;margin:8px 0}li{margin:5px 0}ul.qa li{margin-bottom:10px}
.three{display:grid;grid-template-columns:repeat(3,1fr);gap:14px}.three .card h3{margin:0 0 8px;font-size:16px}.three ul{margin:0;padding-left:18px}
details{margin:10px 0}summary{cursor:pointer;color:#0066B3}
mark{background:#fff3b0;color:inherit;padding:0 3px;border-radius:3px}
code{font-size:12px;background:#f2f6fa;padding:1px 4px;border-radius:4px}
@media(max-width:720px){.grid,.two,.three{grid-template-columns:1fr 1fr}}@media(max-width:480px){.grid,.two,.three{grid-template-columns:1fr}}
@media print{body{background:#fff}section{box-shadow:none;break-inside:avoid}mark{background:none}}
"""

SECTION_TITLES = {"conclusion": "給 AUO 採購主管的結論", "financial": "財報資訊", "conference": "法說會資訊",
                  "sources": "資料來源與限制"}


def esc(s) -> str:
    return html.escape("" if s is None else str(s), quote=True)


def ul(items: list[str], empty: str = NOT_MENTIONED) -> str:
    if not items:
        return f'<ul><li class="na">{esc(empty)}</li></ul>'
    return "<ul>" + "".join(f"<li>{esc(x)}</li>" for x in items) + "</ul>"


def _delta_html(cur: float | None, prev: float | None, kind: str) -> str:
    txt = fmt_delta(cur, prev, kind)
    if cur is None or prev is None:
        return f'<span class="na">{esc(txt)}</span>'
    cls = "delta-positive" if (cur > prev) else ("delta-negative" if cur < prev else "")
    return f'<span class="{cls}">{esc(txt)}</span>'


def render_header(facts: dict, *, generated_on: str, version: str) -> str:
    c = facts["company"]
    period_line = f"{facts['period_label']} 財報"
    if facts.get("reported_period"):
        period_line += f"｜法說報告期間：{facts['reported_period']}"
    rd = esc(facts["report_date"]) if facts.get("report_date") else f'<span class="na">{NOT_AVAILABLE}（財報檔案無 doc_date）</span>'
    cd = esc(facts["conference_date"]) if facts.get("conference_date") else f'<span class="na">{NOT_AVAILABLE}（無場次 / 彙整）</span>'
    return f"""<header>
  <h1>{esc(c.get('company_name'))}｜財報＋法說整合報告</h1>
  <p>{esc(period_line)} ｜ {esc(c.get('market'))}: {esc(c.get('stock_code') or c.get('company_key'))}{('｜' + esc(c.get('industry'))) if c.get('industry') else ''}</p>
  <p>財報公布：{rd}　·　法說會：{cd}</p>
  <p class="mini" style="color:#d8e8f7">產出日期：{esc(generated_on)}｜版本：{esc(version)}｜金額單位：{esc(facts['scale']['prefix'])} {esc(facts['scale']['label'])}（{esc(facts['scale']['note'])}）</p>
</header>"""


def render_conclusion(llm: dict) -> str:
    actions = "".join(f"<li><b>{esc(a['area'])}：</b>{esc(a['advice'] or NOT_MENTIONED)}</li>" for a in llm["procurement_actions"])
    return f"""<section id="conclusion"><h2>{SECTION_TITLES['conclusion']}</h2>
<p>{esc(llm['conclusion']) if llm['conclusion'] else f'<span class="na">{NOT_MENTIONED}（LLM 未產出結論）</span>'}</p>
<div class="two">
  <div class="card"><h3>已證實事實</h3>{ul(llm['confirmed_facts'])}<h3>管理層展望</h3>{ul(llm['management_outlook'])}</div>
  <div class="card"><h3>採購建議</h3><ul>{actions}</ul><p class="mini">※ 不以公司整體財務變動直接推論個別料號的價格；議價前以料號層級成本與供應商確認。</p></div>
</div></section>"""


def merged_segments(facts: dict, llm: dict) -> tuple[list[dict], str | None]:
    """事業群表：爬蟲 summary.business_segments 優先，否則 LLM segments。回 (列, 來源說明)。"""
    if facts.get("summary_segments"):
        return facts["summary_segments"], f"爬蟲法說彙整 business_segments（{facts.get('summary_period') or ''} 場次，LLM 自簡報／新聞稿抽取；值為原文字串）"
    if llm.get("segments"):
        return llm["segments"], "報告 LLM 自本期文件抽取（值為原文字串）"
    return [], None


def render_kpis(facts: dict, segs: list[dict], highlights: list[str]) -> str:
    cards = []
    for k in facts["kpis"]:
        tone = f' class="{k["tone"]}"' if k["tone"] else ""
        cards.append(f'<div class="kpi">{esc(k["label"])}<b>{esc(k["value"])}</b><small{tone}>{esc(k["note"])}</small></div>')
    for s in [s for s in segs if s.get("revenue") or s.get("operating_margin")][:2]:
        if s.get("revenue"):
            cards.append(f'<div class="kpi">{esc(s["segment"])} 營收<b>{esc(s["revenue"])}</b><small>YoY {esc(s.get("revenue_yoy") or "—")}</small></div>')
        else:
            cards.append(f'<div class="kpi">{esc(s["segment"])} 營益率<b>{esc(s["operating_margin"])}</b><small>YoY {esc(s.get("opm_yoy") or "—")}；法說未給營收金額</small></div>')
    while len(cards) < 4:
        cards.append(f'<div class="kpi">事業群營收<b><span class="na">{NOT_AVAILABLE}</span></b><small>財報數據無部門別；法說／文件未提供</small></div>')
    hl = f'<p class="mini" style="margin-top:12px">財報重點（LLM 自文件整理，數字標來源）：</p>{ul(highlights)}' if highlights else ""
    return f'<section><h2>01　財報重點</h2><div class="grid">{"".join(cards[:4])}</div>{hl}</section>'


def render_actuals(facts: dict) -> str:
    a = facts["actuals"]
    cur, prev = a["columns"]
    rows = []
    for m in a["metrics"]:
        cv = (cur["values"] or {}).get(m["key"]) if cur["values"] else None
        pv = (prev["values"] or {}).get(m["key"]) if prev["values"] else None
        rows.append(f"<tr><td>{esc(m['label'])}</td><td>{esc(fmt_value(cv, m['kind']))}</td><td>{esc(fmt_value(pv, m['kind']))}</td>"
                    f"<td>{_delta_html(cv, pv, m['kind'])}</td></tr>")
    note = "" if facts["has_financials"] else f'<div class="note"><b>未取得：</b>此公司目前沒有結構化財報來源（{esc(facts["company"].get("market"))} 市場），實績表、指標表與圖表皆無法產生；財報重點僅依文件文字整理。</div>'
    return f"""<section><h2>02　財報實績</h2>
<div class="scroll"><table><tr><th>指標</th><th>{esc(cur['label'])}</th><th>{esc(prev['label'])}</th><th>同比變動／對照</th></tr>{''.join(rows)}</table></div>
<p class="mini">營業現金流為單季（由累計推算，舊批次無值）；流動比、速動比為資產負債表時點數據；營業利益率的同比以百分點（pt）表示。速動資產＝現金及約當現金＋流動金融資產＋應收票據與帳款；速動比＝速動資產／流動負債。</p>
{note}</section>"""


def render_segments(segs: list[dict], src: str | None) -> str:
    if not segs:
        return f'<section><h2>03　事業群財務表現</h2><p class="na">{NOT_MENTIONED}（財報數據無部門別；法說彙整與文件都沒有事業群數字）</p></section>'
    def cell(v):
        if not v:
            return '<span class="na">—</span>'
        s = esc(v)
        if s.startswith("+"):
            return f'<span class="delta-positive">{s}</span>'
        if s.startswith("−") or s.startswith("-"):
            return f'<span class="delta-negative">{s}</span>'
        return s
    has_note = any(s.get("note") for s in segs)
    rows = "".join(f"<tr><td>{esc(s['segment'])}</td><td>{cell(s.get('revenue'))}</td><td>{cell(s.get('revenue_yoy'))}</td><td>{cell(s.get('operating_margin'))}</td><td>{cell(s.get('opm_yoy'))}</td>"
                   + (f"<td class=\"mini\">{esc(s.get('note') or '')}</td>" if has_note else "") + "</tr>" for s in segs)
    return (f'<section><h2>03　事業群財務表現</h2><div class="scroll"><table><tr><th>事業群</th><th>營收</th><th>營收 YoY</th><th>營益率（法說口徑）</th><th>YoY</th>{"<th>說明</th>" if has_note else ""}</tr>{rows}</table></div>'
            f'<p class="mini">來源：{esc(src)}；值為文件原文，未進財報數據表，不可與第 02 節相減。</p></section>')


def render_trend_charts(facts: dict) -> str:
    scale = facts["scale"]
    parts = []
    for num, title, series, note in (("04", "五年趨勢｜營收 × OPM", facts["annual"], f"年度財報數據；柱為營收（{esc(scale['prefix'])} {esc(scale['label'])}）、折線為營業利益率（右軸 %）。"),
                                     ("05", "五季趨勢｜營收 × OPM", facts["quarterly"], "最近五個季度列；季度與年度口徑不同，不可與第 04 節直接比較。推算列（台股 Q4）無營業利益率，折線斷點。")):
        body = f'<div class="chart">{svg_revenue_opm(series, scale, title=title)}</div>' if series else f'<div class="chart-missing">{NOT_AVAILABLE}</div>'
        parts.append(f'<section><h2>{num}　{esc(title)}</h2>{body}<p class="mini">{note}</p></section>')
    return "\n".join(parts)


def render_indicators(facts: dict) -> str:
    annual = facts["annual"]
    scale = facts["scale"]
    years = "".join(f"<th>{esc(s['label'])}</th>" for s in annual) or "<th>—</th>"
    rows = []
    for ind in INDICATORS:
        label = ind["label"]
        if ind["key"] == "net_margin_parent":
            label += "（%，歸屬母公司）" if not facts.get("net_margin_fallback_years") else "（%）"
        elif ind["kind"] == "money":
            label += f"（{scale['prefix']} {scale['label']}）"
        elif ind["kind"] == "pct":
            label += "（%）"
        elif ind["kind"] == "ratio":
            label += "（x）"
        elif ind["kind"] == "days":
            label += "（days）"
        elif ind["kind"] == "eps":
            label += "（元）"
        cells = "".join(f"<td>{esc(fmt_value(s['values'].get(ind['key']), ind['kind']))}</td>" for s in annual) or f'<td class="na">{NOT_AVAILABLE}</td>'
        rows.append(f"<tr><td>{esc(label)}</td>{cells}</tr>")
    missing = [i["label"] for i in INDICATORS if not facts["indicator_avail"].get(i["key"])]
    notes = []
    if missing:
        notes.append("未取得：" + "、".join(missing) + "（財報數據來源沒有對應欄位，不推算、不畫圖；速動比／應收天數／應付天數需爬蟲 2026-10-21 新版財報數據）")
    if facts.get("net_margin_fallback_years"):
        notes.append("稅後淨利率：" + "、".join(period_label(p) for p in facts["net_margin_fallback_years"]) + " 為合併口徑（歸屬母公司淨利欄位缺），其餘為歸屬母公司。")
    charts = []
    for ind in INDICATORS:
        if ind["key"] == "revenue":
            continue                                   # 第 04 節已有五年營收圖，不重畫
        sub = {"money": f"({scale['prefix']} {scale['label']})", "pct": "(%)", "ratio": "(x)", "eps": "(元)", "days": "(days)"}[ind["kind"]]
        body = (f'<div class="chart">{svg_indicator(annual, ind["key"], scale)}</div>' if facts["indicator_avail"].get(ind["key"])
                else f'<div class="chart-missing">{NOT_AVAILABLE}（財報數據來源無此欄位）</div>')
        charts.append(f'<div class="trend-card" data-indicator="{ind["key"]}"><h3>{esc(ind["label"])} <small>{esc(sub)}</small></h3>{body}</div>')
    yr = f"{annual[0]['label']}–{annual[-1]['label']}" if annual else "—"
    return f"""<section><h2>06　近五年財務指標｜{esc(yr)}</h2>
<div class="scroll"><table><tr><th>指標</th>{years}</tr>{''.join(rows)}</table></div>
<div class="trend-grid">{''.join(charts)}</div>
<p class="mini">公式：毛利率＝毛利／營收；稅前淨利率＝稅前利益／營收；稅後淨利率＝歸屬母公司淨利／營收；負債比＝總負債／總資產；流動比＝流動資產／流動負債；速動比＝速動資產／流動負債；
應收天數＝平均應收帳款／營收×365；應付天數＝平均應付帳款／銷貨成本×365（最早一年用期末值，其餘期初期末平均）。比率欄直接取財報數據來源算好的值。{('<br>' + '<br>'.join(esc(n) for n in notes)) if notes else ''}</p>
</section>"""


def merged_conference(facts: dict, llm: dict) -> dict:
    """法說七節：每節 {text|None, items:[], source}。01–06 爬蟲 management_views 優先、缺時 LLM；07 qa_highlights 優先、缺時 LLM qa。"""
    views = facts.get("summary_views") or {}
    conf = llm["conference"]
    out = {}
    for key, name, _ in CONF_TOPICS:
        if views.get(key):
            out[key] = {"text": views[key], "items": [], "source": f"爬蟲法說彙整 management_views「{name}」"}
        elif conf.get(key):
            out[key] = {"text": None, "items": conf[key], "source": "報告 LLM 自本期文件整理"}
        else:
            out[key] = {"text": None, "items": [], "source": None}
    qa = facts.get("summary_qa") or conf.get("qa") or []
    out["qa"] = {"text": None, "items": qa, "source": ("爬蟲法說彙整 qa_highlights" if facts.get("summary_qa") else ("報告 LLM 自本期文件整理" if conf.get("qa") else None))}
    return out


def render_conference(facts: dict, llm: dict) -> str:
    merged = merged_conference(facts, llm)
    sm = facts.get("summary") or {}
    src_line = (f'<p class="mini">法說資料來源：爬蟲法說彙整（{esc(sm.get("period") or "")} 場次，法說日 {esc(sm.get("conference_date") or "—")}'
                f'{("，報告期間「" + esc(facts["reported_period"]) + "」") if facts.get("reported_period") else ""}）＋ 報告 LLM 讀本期文件'
                + ("；<b>彙整期別與本報告期別不同</b>" if (sm.get("period") and sm.get("period") != facts["period"]) else "") + "</p>"
                if sm else '<p class="mini">法說資料來源：本期沒有爬蟲法說彙整，以下各節由報告 LLM 讀本期文件整理；文件也沒提到的寫「未提及」。</p>')
    blocks = []
    for key, _name, title in CONF_TOPICS:
        m = merged[key]
        if m["text"]:
            body = f"<p>{esc(m['text'])}</p>"
        elif m["items"]:
            body = ul(m["items"])
        else:
            body = f'<p class="na">{NOT_MENTIONED}</p>'
        if key == "profitability" and facts["has_financials"]:
            cur, prev = facts["actuals"]["columns"]
            cv, pv = (cur["values"] or {}), (prev["values"] or {})
            body += (f'<p class="mini"><b>財報對照（財報數據）</b>：營業利益率 {esc(fmt_value(cv.get("operating_margin"), "pct"))}'
                     f'（去年同季 {esc(fmt_value(pv.get("operating_margin"), "pct"))}，{esc(fmt_delta(cv.get("operating_margin"), pv.get("operating_margin"), "pct"))}）、'
                     f'毛利率 {esc(fmt_value(cv.get("gross_margin"), "pct"))}（{esc(fmt_value(pv.get("gross_margin"), "pct"))}）。</p>')
        src = f'<p class="mini">來源：{esc(m["source"])}</p>' if m["source"] else ""
        blocks.append(f'<section><h2>{esc(title)}</h2>{body}{src}</section>')
    qa = merged["qa"]
    if qa["items"]:
        qa_body = '<ul class="qa">' + "".join(f"<li><b>Q：</b>{esc(x['q'])}<br><b>A：</b>{esc(x['a'])}</li>" for x in qa["items"]) + "</ul>"
        qa_body += f'<p class="mini">來源：{esc(qa["source"])}</p>'
    else:
        qa_body = f'<p class="na">{NOT_MENTIONED}（無公開問答；只有電話會議型法說拿得到逐字稿）</p>'
    blocks.append(f'<section><h2>{QA_TITLE}</h2>{qa_body}</section>')
    other = (facts.get("summary_views") or {}).get("_other")
    if other:
        blocks.append(f'<details><summary>爬蟲彙整的其他主題（舊版詞彙，未對應到七節）</summary>{ul(other)}</details>')
    extra = ""
    pos, watch, open_q = ([x for x in (sm.get(k) or []) if x] for k in ("positive_signals", "watch_items", "open_questions"))
    if pos or watch or open_q:
        def card(t, items):
            return f'<div class="card"><h3>{t}</h3>{ul(items, empty="（空）")}</div>'
        extra = (f'<section><h2>08　採購觀察｜正向訊號・觀察項・未答問題 <small class="mini">（模板外補充；來自爬蟲法說彙整）</small></h2>'
                 f'<div class="three">{card("正向訊號", pos)}{card("觀察項", watch)}{card("法說未答／未揭露", open_q)}</div></section>')
    return f'<div class="band" id="conference">{SECTION_TITLES["conference"]}</div>\n{src_line}\n' + "\n".join(blocks) + extra


def render_financial(facts: dict, llm: dict) -> str:
    segs, seg_src = merged_segments(facts, llm)
    return "\n".join([
        f'<div class="band" id="financial">{SECTION_TITLES["financial"]}</div>',
        render_kpis(facts, segs, llm.get("financial_highlights") or []), render_actuals(facts), render_segments(segs, seg_src),
        render_trend_charts(facts), render_indicators(facts),
    ])


def render_sources(facts: dict, llm: dict) -> str:
    items = []
    if facts["has_financials"]:
        src = "、".join(facts["financial_source"]) or "s_ir_calendar_financials"
        urls = "".join(f' <a href="{esc(u)}">[{i + 1}]</a>' for i, u in enumerate(facts["financial_source_urls"][:6]))
        batch = f"（批次 {esc(facts['financial_batch'][0])}）" if facts.get("financial_batch") else ""
        items.append(f"<li>結構化財報數據：{esc(src)}{urls}{batch} — 第 01／02／04／05／06 節全部數字與圖</li>")
    for d in facts["docs"]:
        name = esc(d.get("file_name"))
        link = f'<a href="{esc(d["source_url"])}">{name}</a>' if d.get("source_url") else name
        items.append(f"<li>{link} — {esc(d.get('doc_kind'))}；解析字元 {esc(d.get('text_chars') or 0)}{('；日期 ' + esc(d['doc_date'])) if d.get('doc_date') else ''} — 法說七節、事業群表、結論（LLM 讀文件）</li>")
    sm = facts.get("summary") or {}
    for s in sm.get("sources") or []:
        if isinstance(s, dict) and s.get("url"):
            items.append(f'<li><a href="{esc(s["url"])}">{esc(s.get("title") or s["url"])}</a>｜{esc(s.get("media") or "")} — 爬蟲法說彙整引用</li>')
    if facts["company"].get("ir_url"):
        items.append(f'<li><a href="{esc(facts["company"]["ir_url"])}">公司投資人關係網站</a> — 法說會資料入口</li>')
    limits = [f"金額口徑：{facts['scale']['note']}；比率由財報數據來源算好（%）；年度與季度口徑不同，不可直接比較。"]
    missing = [i["label"] for i in INDICATORS if not facts["indicator_avail"].get(i["key"])]
    if missing:
        limits.append("未取得指標：" + "、".join(missing) + "（來源無對應欄位，不推算）。")
    if not facts["has_financials"]:
        limits.append("此公司無結構化財報來源，所有數字表格與圖表為「未取得」。")
    if sm.get("period") and sm.get("period") != facts["period"]:
        limits.append(f"法說彙整來自 {sm['period']} 場次（本期 {facts['period']} 沒有彙整），引用時注意期別。")
    if sm.get("notes"):
        limits.append(f"爬蟲彙整備註：{sm['notes']}")
    limits.append("第 01／02 節與五季圖的淨利為合併淨利（含非控制權益），第 06 節稅後淨利率為歸屬母公司；法說節文字為 LLM 彙整（含媒體轉述），正式決策前仍應以公開資訊觀測站／SEC 及公司合併財報附註為準。")
    return f"""<section id="sources"><h2>{SECTION_TITLES['sources']}</h2>
<ul>{''.join(items) or '<li class="na">無</li>'}</ul>
<h3>LLM 實際引用</h3>{ul(llm['sources_used'], empty='未標示')}
<div class="note"><b>限制：</b>{'<br>'.join(esc(x) for x in limits + llm['caveats'])}</div>
</section>"""


def render_report(facts: dict, llm: dict, *, generated_on: str, version: str) -> str:
    """完整 HTML。generated_on = 'YYYY-MM-DD'；version 例 'template 20261021 / prompt 2.0'。"""
    c = facts["company"]
    title = f"{c.get('company_name')}（{c.get('stock_code') or c.get('company_key')}）{facts['period_label']} 財報＋法說整合報告"
    return f"""<!doctype html>
<html lang="zh-Hant-TW">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>{esc(title)}</title>
<meta name="generator" content="ir_calendar_build_supplier_report {esc(version)}">
<style>
{TEMPLATE_CSS}</style>
</head>
<body>
{render_header(facts, generated_on=generated_on, version=version)}
<main>
{render_conclusion(llm)}
{render_financial(facts, llm)}
{render_conference(facts, llm)}
{render_sources(facts, llm)}
<footer class="mini">{esc(title)}｜{esc(generated_on)}｜{esc(version)}｜AUO 採購內部使用，非投資建議</footer>
</main>
</body>
</html>
"""


def report_file_name(facts: dict) -> str:
    slug = facts["company"].get("company_slug") or facts["company"].get("company_key", "unknown").replace(":", "_")
    return f"{slug}_{facts['period']}_supplier_report.html"


In [ ]:
# [c08] llm_client
# LLM 接線：Azure Foundry 的 Anthropic Messages 端點，與 meeting_extracted_w1_t2_extract [c06] / sales_report_mail 同一組 secret / URL。
# requests 是 DBR 內建，不 %pip install。thinking 關閉、只要 JSON；system 開 prompt caching（同一份 system prompt 多家公司共用）。
# 不送 temperature：claude-sonnet-5 回 400「temperature is deprecated」（2026-10-16 實測）。
# 重試：HTTP 408 / 429 / 5xx / 連線錯誤指數退避 3 次；其他 4xx 直接回 error。JSON 失敗的重試在 [c11]。
import requests

FOUNDRY_URL = "https://micenter.services.ai.azure.com/anthropic/v1/messages"
FOUNDRY_API_KEY = dbutils.secrets.get(scope="llm-api-keys", key="ms-foundry")
HTTP_TIMEOUT_S = 300          # 輸入可達 10 萬字，給長一點


def call_llm(messages: list[dict], system_prompt: str, model: str, *, cache: bool = True,
             max_tokens: int = LLM_MAX_TOKENS) -> dict:
    """回 {"text", "stop_reason", "usage": {input, output, cache_read, cache_write}, "latency_s", "error"}；
    error 非 None 代表端點層失敗（已重試），text 為 None。"""
    system = [{"type": "text", "text": system_prompt}]
    if cache:
        system[0]["cache_control"] = {"type": "ephemeral"}
    body = {"model": model, "max_tokens": max_tokens, "system": system, "messages": messages,
            "thinking": {"type": "disabled"}}
    headers = {"x-api-key": FOUNDRY_API_KEY, "anthropic-version": "2023-06-01", "content-type": "application/json"}
    t0 = time.time()
    err = None
    for attempt in range(3):
        try:
            resp = requests.post(FOUNDRY_URL, headers=headers, json=body, timeout=HTTP_TIMEOUT_S)
            if resp.status_code in (408, 429) or resp.status_code >= 500:
                err = f"HTTP {resp.status_code}: {resp.text[:200]}"
            elif resp.status_code >= 400:
                return {"text": None, "stop_reason": None, "usage": {}, "latency_s": round(time.time() - t0, 2),
                        "error": f"HTTP {resp.status_code}: {resp.text[:300]}"}
            else:
                data = resp.json()
                text = "\n".join(b.get("text", "") for b in data.get("content", []) if b.get("type") == "text")
                u = data.get("usage") or {}
                usage = {"input": u.get("input_tokens", 0) or 0, "output": u.get("output_tokens", 0) or 0,
                         "cache_read": u.get("cache_read_input_tokens", 0) or 0,
                         "cache_write": u.get("cache_creation_input_tokens", 0) or 0}
                return {"text": text, "stop_reason": data.get("stop_reason"), "usage": usage,
                        "latency_s": round(time.time() - t0, 2), "error": None}
        except (requests.RequestException, ValueError) as e:
            err = f"{type(e).__name__}: {str(e)[:200]}"
        time.sleep(5 * (2 ** attempt))
    return {"text": None, "stop_reason": None, "usage": {}, "latency_s": round(time.time() - t0, 2), "error": err}


print("llm client ready:", FOUNDRY_URL.split("/anthropic")[0], "| model:", settings["model"])


In [ ]:
# [c09] io_load
# 讀 silver，把一個工作項（公司 × 期別）需要的列 collect 到 driver 組 facts。每家公司：financials ≤ 幾十列、文件 ≤ 幾份，
# collect 可接受；文件全文（text_md）只取該公司該期、最多 max_total_chars 字進 prompt。
# summary：先取同期別；沒有時退用該公司最近一場（period 較小者取最大，避免 10 月法說標 Q3 但 Q2 財報期沒有彙整），facts 會標示期別不同。


def table_name(settings: dict, layer: str, short: str) -> str:
    return f"{settings['catalog']}.{settings['schema']}.{LAYER_PREFIX[layer]}{settings['domain']}_{short}"


def _rows(df: DataFrame) -> list[dict]:
    return [r.asDict(recursive=True) for r in df.collect()]


def _stringify_dates(d: dict | None, keys: tuple[str, ...]) -> dict | None:
    if d:
        for k in keys:
            if d.get(k) is not None:
                d[k] = str(d[k])
    return d


def load_company_bundle(settings: dict, work: dict) -> dict:
    """work = select_work 的一列（company_key / period / …）。回 {company, fin_rows, summary, conference, docs}。"""
    ck, period = work["company_key"], work["period"]
    s = settings
    company_rows = _rows(latest_per_key(spark.table(table_name(s, "silver", SRC_COMPANY)).filter(F.col("company_key") == ck),
                                        ["company_key"]))
    company = company_rows[0] if company_rows else {"company_key": ck}
    company.setdefault("company_slug", work.get("company_slug"))
    company.setdefault("category", work.get("category"))
    fin_rows = _rows(spark.table(table_name(s, "silver", SRC_FINANCIALS)).filter(F.col("company_key") == ck))
    df_sm = spark.table(table_name(s, "silver", SRC_SUMMARY)).filter(F.col("company_key") == ck)
    sm_rows = _rows(df_sm.filter(F.col("period") == period))
    if not sm_rows:
        sm_rows = _rows(df_sm.filter(F.col("period") < period).orderBy(F.col("period").desc()).limit(1))
    conf_rows = _rows(latest_per_key(spark.table(table_name(s, "silver", SRC_CONFERENCE))
                                     .filter((F.col("company_key") == ck) & (F.col("period") == period)),
                                     ["company_key", "period"], order_col="revision"))
    df_text = (spark.table(table_name(s, "silver", SRC_DOC_TEXT))
               .filter((F.col("company_key") == ck) & (F.col("period") == period) & F.col("parse_status").isin(*DOC_STATUS_OK)))
    df_text = latest_per_key(df_text, ["volume_path"], order_col="parsed_at")
    df_file = spark.table(table_name(s, "silver", SRC_DOC_FILE)).select("volume_path", "source_url", "doc_date")
    docs = _rows(df_text.join(df_file, "volume_path", "left")
                 .select("volume_path", "file_name", "doc_kind", "sha256", "source_url", "text_chars", "text_md", "fiscal_label", "doc_date"))
    for d in docs:
        _stringify_dates(d, ("doc_date",))
    return {"company": company, "fin_rows": fin_rows,
            "summary": _stringify_dates(sm_rows[0], ("conference_date",)) if sm_rows else None,
            "conference": _stringify_dates(conf_rows[0], ("conference_date",)) if conf_rows else None, "docs": docs}


def build_bundle_facts(bundle: dict, period: str) -> dict:
    return build_facts(bundle["company"], period, bundle["fin_rows"], bundle["summary"], bundle["conference"], bundle["docs"])


In [ ]:
# [c10] io_write
#   write_llm_row  ：bronze b_{domain}_supplier_report_llm，每次呼叫一列 append-only（原始回應 payload 原文，失敗也記）
#   write_report   ：gold  g_{domain}_supplier_report，鍵 (company_key, period) MERGE：同一期重做就覆蓋成最新版
#   write_html     ：Volume <volume_root>/ir_calendar/reports/<分類小寫>/<公司代稱>/<檔名>.html（FUSE 路徑，同名覆蓋）
# 表由 ir_calendar_init_tables [c19a] / [c19b] 建；不存在直接拋錯。dry_run 不會走到這裡。
#   2026-10-21 換新版面後 light / score 一律 NULL（gold 欄位保留，DDL 不動）。
import os

from delta.tables import DeltaTable
from pyspark.sql.types import (ArrayType, DoubleType, IntegerType, LongType, StringType, StructField, StructType,
                               TimestampType)

LLM_SCHEMA = StructType([
    StructField("company_key", StringType()), StructField("period", StringType()),
    StructField("prompt_version", StringType()), StructField("model", StringType()),
    StructField("attempt", IntegerType()), StructField("docs_digest", StringType()),
    StructField("status", StringType()), StructField("error", StringType()), StructField("stop_reason", StringType()),
    StructField("prompt_chars", IntegerType()), StructField("input_tokens", IntegerType()),
    StructField("output_tokens", IntegerType()), StructField("cache_read_tokens", IntegerType()),
    StructField("latency_s", DoubleType()), StructField("payload", StringType()),
    StructField("called_at", TimestampType()), StructField("job_run_id", StringType()),
])
REPORT_SCHEMA = StructType([
    StructField("company_key", StringType()), StructField("period", StringType()),
    StructField("company_name", StringType()), StructField("english_name", StringType()),
    StructField("stock_code", StringType()), StructField("market", StringType()), StructField("category", StringType()),
    StructField("company_slug", StringType()), StructField("conference_date", StringType()),
    StructField("report_title", StringType()), StructField("template_version", StringType()),
    StructField("prompt_version", StringType()), StructField("model", StringType()),
    StructField("docs_digest", StringType()), StructField("doc_count", IntegerType()),
    StructField("doc_kinds", ArrayType(StringType())), StructField("has_financials", StringType()),
    StructField("annual_periods", IntegerType()), StructField("quarterly_periods", IntegerType()),
    StructField("light", StringType()), StructField("score", IntegerType()),
    StructField("llm_json", StringType()), StructField("html", StringType()), StructField("html_path", StringType()),
    StructField("html_chars", LongType()), StructField("input_tokens", IntegerType()),
    StructField("output_tokens", IntegerType()), StructField("generated_at", TimestampType()),
    StructField("job_run_id", StringType()),
])
REPORT_KEYS = ["company_key", "period"]


def write_llm_row(settings: dict, row: dict) -> None:
    table = table_name(settings, "bronze", LLM_TABLE)
    spark.createDataFrame([row], schema=LLM_SCHEMA).write.format("delta").mode("append").saveAsTable(table)


def write_report(settings: dict, row: dict) -> None:
    table = table_name(settings, "gold", REPORT_TABLE)
    df = spark.createDataFrame([row], schema=REPORT_SCHEMA)
    cond = " AND ".join(f"t.{k} <=> s.{k}" for k in REPORT_KEYS)
    (DeltaTable.forName(spark, table).alias("t").merge(df.alias("s"), cond)
     .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())


def html_volume_path(settings: dict, facts: dict) -> str:
    cat = (facts["company"].get("category") or "uncategorized").lower()
    slug = facts["company"].get("company_slug") or facts["company"]["company_key"].replace(":", "_")
    return f"{settings['volume_root']}/{REPORT_DIR}/{cat}/{slug}/{report_file_name(facts)}"


def write_html(path: str, content: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write(content)


def report_row(settings: dict, work: dict, facts: dict, llm: dict, html_text: str, html_path: str,
               usage: dict, now: datetime) -> dict:
    c = facts["company"]
    return {
        "company_key": work["company_key"], "period": work["period"],
        "company_name": c.get("company_name"), "english_name": c.get("english_name"), "stock_code": c.get("stock_code"),
        "market": c.get("market"), "category": c.get("category"), "company_slug": c.get("company_slug"),
        "conference_date": facts.get("conference_date"),
        "report_title": f"{c.get('company_name')}（{c.get('stock_code') or c.get('company_key')}）{facts['period_label']} 財報＋法說整合報告",
        "template_version": TEMPLATE_VERSION, "prompt_version": settings["prompt_version"], "model": settings["model"],
        "docs_digest": work["docs_digest"], "doc_count": int(work["doc_count"]), "doc_kinds": list(work["doc_kinds"] or []),
        "has_financials": "true" if facts["has_financials"] else "false",
        "annual_periods": len(facts["annual"]), "quarterly_periods": len(facts["quarterly"]),
        "light": None, "score": None,      # 2026-10-21 新版面無燈號 / 分數；欄位保留（gold DDL 不動），一律 NULL
        "llm_json": json.dumps(llm, ensure_ascii=False), "html": html_text, "html_path": html_path,
        "html_chars": len(html_text), "input_tokens": int(usage.get("input", 0)), "output_tokens": int(usage.get("output", 0)),
        "generated_at": now, "job_run_id": settings["job_run_id"],
    }



In [ ]:
# [c11] run_once
# 一次執行：工作清單（[c04]）→ 每份報告：撈 silver 組 facts（[c09]）→ 組 prompt、呼叫 LLM（[c05]/[c08]）→ normalize →
# 套版（[c07]，法說七節與事業群表優先用爬蟲彙整、LLM 補位）→ 寫 bronze / gold / Volume（[c10]）。報告數受 max_reports 限制，ThreadPool 併發 concurrency 份。
# JSON 失敗：把上一輪輸出當 assistant turn、帶錯誤訊息再試 1 次；被 max_tokens 截斷改附 COMPACT_HINT 重打。端點失敗就記 failed 列、不產報告。
# render_only：不呼叫 LLM，拿 gold 既有 llm_json 重新套版（版型 / 圖表改了用）。
from concurrent.futures import ThreadPoolExecutor, as_completed

assert PROMPT_VERSION_IN_CELL == settings["prompt_version"], (
    f"[c05] prompt 版本 {PROMPT_VERSION_IN_CELL} ≠ widget prompt_version {settings['prompt_version']}")

VERSION_TAG = f"template {TEMPLATE_VERSION} / prompt {PROMPT_VERSION_IN_CELL}"


def _llm_row(work: dict, settings: dict, attempt: int, r: dict, prompt_chars: int, status: str, error: str | None,
             now: datetime) -> dict:
    u = r.get("usage") or {}
    return {"company_key": work["company_key"], "period": work["period"], "prompt_version": settings["prompt_version"],
            "model": settings["model"], "attempt": attempt, "docs_digest": work["docs_digest"], "status": status,
            "error": (error or "")[:2000] or None, "stop_reason": r.get("stop_reason"), "prompt_chars": prompt_chars,
            "input_tokens": int(u.get("input", 0)), "output_tokens": int(u.get("output", 0)),
            "cache_read_tokens": int(u.get("cache_read", 0)), "latency_s": float(r.get("latency_s") or 0.0),
            "payload": r.get("text"), "called_at": now, "job_run_id": settings["job_run_id"]}


def generate_with_llm(settings: dict, work: dict, facts: dict, docs: list[dict]) -> tuple[dict | None, dict, str | None]:
    """回 (normalized_llm_output | None, usage_total, error)。每次呼叫都寫一列 bronze。"""
    packed, skipped = pack_documents(docs, max_doc_chars=settings["max_doc_chars"], max_total_chars=settings["max_total_chars"])
    user_prompt = build_user_prompt(facts, packed, skipped)
    messages = [{"role": "user", "content": user_prompt}]
    usage_tot = {"input": 0, "output": 0}
    r1 = call_llm(messages, SYSTEM_PROMPT, settings["model"])
    for k in usage_tot:
        usage_tot[k] += (r1.get("usage") or {}).get(k, 0)
    if r1["error"]:
        write_llm_row(settings, _llm_row(work, settings, 1, r1, len(user_prompt), "failed", r1["error"], datetime.now(UTC)))
        return None, usage_tot, r1["error"]
    truncated = r1["stop_reason"] == "max_tokens"
    obj, perr = (None, "output truncated at max_tokens") if truncated else parse_llm_json(r1["text"])
    write_llm_row(settings, _llm_row(work, settings, 1, r1, len(user_prompt), "success" if obj else "json_error", perr, datetime.now(UTC)))
    if obj is None:
        if truncated:
            messages = [{"role": "user", "content": user_prompt + COMPACT_HINT}]
        else:
            messages += [{"role": "assistant", "content": (r1["text"] or "").strip() or "(empty)"},
                         {"role": "user", "content": RETRY_HINT.format(err=perr)}]
        r2 = call_llm(messages, SYSTEM_PROMPT, settings["model"])
        for k in usage_tot:
            usage_tot[k] += (r2.get("usage") or {}).get(k, 0)
        if r2["error"]:
            write_llm_row(settings, _llm_row(work, settings, 2, r2, len(user_prompt), "failed", r2["error"], datetime.now(UTC)))
            return None, usage_tot, r2["error"]
        obj, perr = (None, "output truncated at max_tokens (retry)") if r2["stop_reason"] == "max_tokens" else parse_llm_json(r2["text"])
        write_llm_row(settings, _llm_row(work, settings, 2, r2, len(user_prompt), "success" if obj else "json_error", perr, datetime.now(UTC)))
        if obj is None:
            return None, usage_tot, f"LLM JSON 兩次都失敗：{perr}"
    return normalize_report(obj), usage_tot, None


def process_work(settings: dict, work: dict, existing_llm_json: str | None = None) -> dict:
    """一份報告。回 {work, ok, error, html_path, usage}。"""
    now = datetime.now(UTC)
    bundle = load_company_bundle(settings, work)
    facts = build_bundle_facts(bundle, work["period"])
    if settings["render_only"]:
        if not existing_llm_json:
            return {"work": work, "ok": False, "error": "render_only 但 gold 沒有既有 llm_json", "html_path": None, "usage": {}}
        llm, usage, err = normalize_report(json.loads(existing_llm_json)), {}, None
    else:
        llm, usage, err = generate_with_llm(settings, work, facts, bundle["docs"])
    if err:
        return {"work": work, "ok": False, "error": err, "html_path": None, "usage": usage}
    html_text = render_report(facts, llm, generated_on=now.strftime("%Y-%m-%d"), version=VERSION_TAG)
    path = html_volume_path(settings, facts)
    write_html(path, html_text)
    write_report(settings, report_row(settings, work, facts, llm, html_text, path, usage, now))
    return {"work": work, "ok": True, "error": None, "html_path": path, "usage": usage}


def run_once(settings: dict) -> dict:
    df_text = spark.table(table_name(settings, "silver", SRC_DOC_TEXT))
    df_report = spark.table(table_name(settings, "gold", REPORT_TABLE))
    df_work = select_work(df_text, df_report, categories=settings["categories"], company_keys=settings["company_keys"],
                          periods=settings["periods"], prompt_version=settings["prompt_version"],
                          template_version=TEMPLATE_VERSION, force=settings["force_rebuild"] or settings["render_only"])
    work = [r.asDict() for r in df_work.orderBy("category", "company_key", "period").limit(settings["max_reports"]).collect()]
    report = {"work": len(work), "ok": 0, "errors": [], "paths": [], "usage": {"input": 0, "output": 0}}
    print(f"待產報告 {len(work)} 份（max_reports={settings['max_reports']} categories={settings['categories']} "
          f"company_keys={settings['company_keys']} periods={settings['periods']} force={settings['force_rebuild']} "
          f"render_only={settings['render_only']}）")
    for w in work:
        print(f"  {w['category']:<12} {w['company_key']:<12} {w['company_slug'] or '':<24} {w['period']}  {w['doc_count']} 檔 {list(w['doc_kinds'] or [])}")
    if not work:
        return report
    if settings["dry_run"]:
        for w in work:
            bundle = load_company_bundle(settings, w)
            facts = build_bundle_facts(bundle, w["period"])
            packed, skipped = pack_documents(bundle["docs"], max_doc_chars=settings["max_doc_chars"], max_total_chars=settings["max_total_chars"])
            prompt = build_user_prompt(facts, packed, skipped)
            views = facts.get("summary_views") or {}
            print(f"  [dry_run] {w['company_key']} {w['period']}: financials annual={len(facts['annual'])} quarterly={len(facts['quarterly'])} "
                  f"速動比={'有' if facts['indicator_avail'].get('quick_ratio') else '無'} "
                  f"summary={facts.get('summary_period') or '無'}（views {len([k for k in views if k != '_other'])}/6、segments {len(facts.get('summary_segments') or [])}、"
                  f"qa {len(facts.get('summary_qa') or [])}）docs={len(packed)}（截尾 {sum(d['truncated'] for d in packed)}，略過 {skipped}）"
                  f"prompt_chars={len(prompt)} → {html_volume_path(settings, facts)}")
        return report

    existing = {}
    if settings["render_only"]:
        keys = [(w["company_key"], w["period"]) for w in work]
        for r in df_report.select("company_key", "period", "llm_json").collect():
            if (r["company_key"], r["period"]) in keys:
                existing[(r["company_key"], r["period"])] = r["llm_json"]

    results = []
    with ThreadPoolExecutor(max_workers=settings["concurrency"]) as ex:
        futs = {ex.submit(process_work, settings, w, existing.get((w["company_key"], w["period"]))): w for w in work}
        for fut in as_completed(futs):
            w = futs[fut]
            try:
                res = fut.result()
            except Exception as e:  # noqa: BLE001 - 一份失敗不拖垮整批，記錯誤繼續
                res = {"work": w, "ok": False, "error": repr(e)[:500], "html_path": None, "usage": {}}
            results.append(res)
            tag = "ok    " if res["ok"] else "FAILED"
            print(f"  {tag} {w['company_key']:<12} {w['period']}  {res['html_path'] or res['error']}")
    for res in results:
        if res["ok"]:
            report["ok"] += 1
            report["paths"].append(res["html_path"])
        else:
            report["errors"].append(f"{res['work']['company_key']} {res['work']['period']}: {res['error']}")
        for k in report["usage"]:
            report["usage"][k] += (res.get("usage") or {}).get(k, 0)
    print(f"完成：{report['ok']} / {len(work)} 份；tokens in={report['usage']['input']} out={report['usage']['output']}；失敗 {len(report['errors'])}")
    return report



In [ ]:
# [c20] main
# job 進入點。有失敗的報告且 fail_on_error = true 才 raise（讓 Databricks job 顯示失敗、通知才會發）。
result = run_once(settings)
print(json.dumps({k: v for k, v in result.items() if k != "paths"}, ensure_ascii=False))
for p in result["paths"]:
    print("  ", p)
if result["errors"] and settings["fail_on_error"]:
    raise RuntimeError(f"{len(result['errors'])} 份報告失敗：" + " | ".join(result["errors"])[:2000])


In [ ]:
# [c30] check_target
# 寫入後驗證（開發用）：gold 鍵唯一、最近幾份報告的摘要、HTML 沒有殘留模板待填欄位 {{…}}。
df_g = spark.table(table_name(settings, "gold", REPORT_TABLE))
n_dup = df_g.groupBy("company_key", "period").count().filter("count > 1").count()
assert n_dup == 0, "g_supplier_report (company_key, period) 重複"
recent = (df_g.select("company_key", "period", "company_name", "has_financials", "doc_count",
                      "html_chars", "prompt_version", "template_version", "generated_at",
                      F.col("html").rlike(r"\{\{[^}]+\}\}").alias("has_placeholder"))
          .orderBy(F.col("generated_at").desc()).limit(20))
for r in recent.collect():
    print(f"{r.generated_at:%Y-%m-%d %H:%M}  {r.company_key:<12} {r.period}  {r.company_name or '':<16} "
          f"fin={r.has_financials}  docs={r.doc_count}  html={r.html_chars}  "
          f"{r.template_version}/{r.prompt_version}  placeholder={r.has_placeholder}")
n_bad = recent.filter("has_placeholder").count()
assert n_bad == 0, f"{n_bad} 份報告 HTML 殘留 {{{{…}}}} 待填欄位"
print(f"rows={df_g.count()} dup_keys={n_dup}")

